# 14 — Diagnóstico automático do limiar de confiança do B_e03

A auditoria por IA encontrou problemas de rótulo e indícios de baixa confiança:
34 dos 36 alertas perdidos pelo B em relação ao candidato 10 tinham registro
de proposta abaixo de 0,25. Agora vamos medir o que acontece ao variar esse corte.

**Sem novo treinamento, sem preenchimento de fichas e sem alterar rótulos.**
O notebook gera as comparações, verifica os requisitos e salva um pacote final.

- Pesos fixos: `B_home_replay_e03.pt`, autenticados pelo hash do experimento 12.
- Imagens: as mesmas **456 Home VAL + 1.000 Open Images VAL**.
- Grade global fixa: **0,05 / 0,10 / 0,15 / 0,20 / 0,25 / 0,30 / 0,40**.
- Uma inferência por imagem em 0,001; os sete pontos reutilizam esse cache.
- Baseline e candidato 10: referências autenticadas do pacote 12, em 0,25.
- Reprodução do B em 0,25 obrigatória antes de sugerir um limiar.

As 300 reservadas e os testes ficam sem extração/inferência. As validações
já participaram do desenvolvimento; esta etapa não é um teste final independente.

**Versão corrigida em 16/10/2026: checagem de cache 1.0.1.**


## 1. Executar no Kaggle

Importe este arquivo, ative Internet e habilite o Secret existente
**GDRIVE_RCLONE_CONFIG_B64**. Prefira **GPU T4**; CPU também é aceita,
com inferência mais demorada. Use **Save Version → Save & Run All**.

Os caminhos do Drive estão preenchidos. Não é necessário importar os
pacotes anteriores como inputs, preencher o revisor 13 ou selecionar um limiar.
Somente os dados das validações são extraídos; não há download dos lotes de treino.

Ao término, os resultados estão em **pacote_diagnostico_14.zip**. Os relatórios e os caches
também serão copiados para a pasta da execução no Drive.


In [ ]:
%pip install -q ultralytics==8.4.150 pyyaml pillow scipy matplotlib


In [ ]:
import os
os.environ['YOLO_OFFLINE']='true'
import sys, json, hashlib, shutil, platform, random, inspect
from pathlib import Path
from datetime import datetime,timezone
import numpy as np
import PIL, scipy, torch, ultralytics
from ultralytics import YOLO,settings
from IPython.display import display,FileLink,HTML

SEED=42
random.seed(SEED);np.random.seed(SEED);torch.manual_seed(SEED)
if torch.cuda.is_available():torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic=True
torch.backends.cudnn.benchmark=False
DEVICE=0 if torch.cuda.is_available() else 'cpu'
if ultralytics.__version__!='8.4.150':
    raise RuntimeError('A versão esperada é ultralytics==8.4.150. Reinicie a sessão e execute desde o começo.')
settings.update({k:False for k in ['sync','hub','platform','wandb','comet','mlflow','neptune','clearml','dvc','tensorboard'] if k in settings})
NOME_EXPERIMENTO='diagnostico14_'+datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S_%f')
PASTA_LOCAL=Path('/tmp/tcc_confianca_14')/NOME_EXPERIMENTO
PASTA_RESULTADOS=Path('/kaggle/working/resultados_tcc')/NOME_EXPERIMENTO
PASTA_LOCAL.mkdir(parents=True,exist_ok=True)
PASTA_RESULTADOS.mkdir(parents=True,exist_ok=True)
PASTA_EXECUCAO_11='resultados/diagnostico11_20261016_024603_683411'
PASTA_EXECUCAO_12='resultados/ajuste12_20261016_033819_596490'
AMBIENTE={'python':platform.python_version(),'torch':torch.__version__,
    'ultralytics':ultralytics.__version__,'numpy':np.__version__,'pillow':PIL.__version__,
    'scipy':scipy.__version__,'device':str(DEVICE),
    'gpu':torch.cuda.get_device_name(0) if DEVICE==0 else None}
print('Dispositivo:',AMBIENTE['gpu'] or 'CPU')
print('Diagnóstico automático: sem treino e sem formulários.')


In [ ]:
EXPECTED = {'baseline': '4170653e0c0a604f85b6747e42b3fd2ff5ddb765ee4762915dfc9f3d33d322d9', 'candidato': '97040f91876d8835a67a7db287d3e1c5f129a4068e36dd4483d947f62b260c5f', 'politica': '1aa6b13cb68d8f30a2f388c3577a61a12dcac4c604581877091d3fbdc6042845', 'home_zip': 'ada7fbee37b83f673b1b76bc57643d1fe4648941e0e8a7b895282216aed47676', 'test_home': 'f48be0a2b3a93e59259b1ae0d3655028951bf270ef7f4e7916a9424530951a89', 'preparacao': '9865042654c51e83955c2aeb186a1b5e27e1028a5c56f1f163e2891abc0bec49', 'divisao_10': 'a5dd35ff484dda77e8988aefdc961f05d07ea409d1336cbe90c4a5420632bab9', 'manifesto_pacotes': '655094dea28755a7bddc817ae66660d06fab28ccb8f9d140d3ffebb07f03a0d4', 'pacote_11': 'a3f3fb500cef83fc4890f6bbec6725bbf5e080503fd0478af3dd84d6f5ef6e93', 'dataset_novo': '695b445e15c905642e2b276801b71132ced28e5eaee92d9fbcdd608f582f15a0', 'zip_novo': '5af596546e2ad734c7b663f4567de2087d7fb42b41527d4852affdb4e7d2b96c', 'codigo_11': {'diag11_core.py': '29e7c52fced44ab4898edfbc46477176587f290d2fbdacad4de5f595d4187670', 'diag11_data.py': 'c9f137d4dcc8c8f5fb156aaf25cddf6707d24e74567c05237332c2144e6e2f98', 'diag11_reports.py': '0068af8fd70b616cb642e9fb8a75d503cdc003f18fe79d64fd487bd7822d85e2'}, 'pacote_12': '1c3a6016099c2bb23aca29396286497c7973f00136c1afbb0bbc62e45aa20650', 'codigo_12': {'diag11_core.py': '29e7c52fced44ab4898edfbc46477176587f290d2fbdacad4de5f595d4187670', 'diag11_data.py': 'c9f137d4dcc8c8f5fb156aaf25cddf6707d24e74567c05237332c2144e6e2f98', 'diag11_reports.py': '0068af8fd70b616cb642e9fb8a75d503cdc003f18fe79d64fd487bd7822d85e2', 'exp10_partial.py': '86a36f93aa643853f1268dcc133bed7e7344dea269b9d1d58cf54095538fffae', 'exp12_protocol.py': '257d09b0b2aafe94da28bdcb59859778ffb31e25601db75d1058092b943b5759', 'exp12_data.py': '0a9de7e99d302a2690f18f51924f4dc80b32861c9d2149ccfe0572ec75499f30', 'exp12_training.py': '6c51c9965336b2f00bd9b282e2159c0b63f07c5c16859da78da2818bd4202fdc', 'exp12_reports.py': '120609247939189f895c01cdfd1411e4d2b416d1e293cb26851d63d04835b49a'}, 'B_e03': 'b683e1ef5fac4f8ddb4fad2f610e10dcb33bc658ea72593b8088652dbdd3c21c', 'regra_12_sha256': '04c6287c2c4adb8c43baf28c10acbaa1a599198b905d48808a8f5f67f39a4909'}
LOCALIZACAO_11 = {'pasta_drive': 'datasets/validacao_independente_11_v1/695b445e15c905642e2b276801b71132ced28e5eaee92d9fbcdd608f582f15a0', 'arquivo': 'validacao_independente_11_695b445e15c9.zip', 'arquivo_sha256': '5af596546e2ad734c7b663f4567de2087d7fb42b41527d4852affdb4e7d2b96c', 'dataset_sha256': '695b445e15c905642e2b276801b71132ced28e5eaee92d9fbcdd608f582f15a0', 'validacao': 1000, 'reserva_futura': 300}
AUDITORIA_13_AUTOMATICA = {'versao': '13-consolidacao-automatica-1.0', 'status': 'analise_automatica_consolidada', 'casos_avaliados_automaticamente': 108, 'revisao_humana_realizada': False, 'preenchimento_manual_obrigatorio': False, 'triagem': {'fora_do_recorte_conservador_proposto': 44, 'objeto_em_foco_compativel_com_recorte_proposto': 28, 'somente_diagnostico_validacao': 36}, 'novo_dataset_pronto': False, 'rotulos_alterados': False, 'pacote_automatico_sha256': '30b560e47e6db7c358396b6872832b20fd99d68c89edc10467b9984ea040fd12'}


## 2. Carregar o avaliador e os extratores

Os módulos originais dos notebooks 11 e 12 são conferidos pelos hashes.
O novo avaliador recebe a confiança como argumento, preservando a política
V2, a associação por IoU e as máscaras de cobertura por imagem/classe.
As caixas e as classes permanecem as originais para permitir comparação.


In [ ]:
MODULOS_EMBUTIDOS = {'diag11_core.py': '"""Núcleo do diagnóstico 11. Não contém treinamento nem seleção de pesos."""\nfrom __future__ import annotations\n\nimport csv\nimport gzip\nimport hashlib\nimport io\nimport json\nimport math\nfrom collections import Counter, defaultdict\nfrom pathlib import Path, PurePosixPath\n\nimport numpy as np\nfrom PIL import Image\n\nNAMES = ["bed", "sofa", "chair", "table", "lamp", "tv", "laptop",\n         "wardrobe", "window", "door", "potted plant", "photo frame"]\nALL_KNOWN = (1 << len(NAMES)) - 1\nALERT_IDS = {0, 1, 2, 3, 7, 10}\nALERT_MASK = sum(1 << c for c in ALERT_IDS)\nLEVEL = {"sem_prioridade": 0, "baixo": 1, "medio": 2, "alto": 3}\n\n\ndef canonical(x):\n    return json.dumps(x, sort_keys=True, ensure_ascii=False, separators=(",", ":"))\n\n\ndef sha256_file(path):\n    h = hashlib.sha256()\n    with Path(path).open("rb") as f:\n        for block in iter(lambda: f.read(2**20), b""):\n            h.update(block)\n    return h.hexdigest()\n\n\ndef json_clean(value):\n    if isinstance(value, dict):\n        return {str(k): json_clean(v) for k, v in value.items()}\n    if isinstance(value, (list, tuple)):\n        return [json_clean(v) for v in value]\n    if isinstance(value, np.generic):\n        return json_clean(value.item())\n    if isinstance(value, Path):\n        return str(value)\n    if isinstance(value, float) and not math.isfinite(value):\n        return None\n    return value\n\n\ndef save_json(path, value):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    tmp = path.with_name(path.name + ".tmp")\n    tmp.write_text(json.dumps(json_clean(value), ensure_ascii=False, indent=2,\n                              allow_nan=False), encoding="utf-8")\n    tmp.replace(path)\n\n\ndef save_csv(path, rows, fields=None):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    rows = list(rows)\n    if fields is None:\n        fields = list(dict.fromkeys(k for row in rows for k in row))\n    with path.open("w", newline="", encoding="utf-8-sig") as f:\n        writer = csv.DictWriter(f, fieldnames=fields, extrasaction="raise")\n        writer.writeheader()\n        writer.writerows(rows)\n\n\ndef safe_member(name):\n    p = PurePosixPath(name)\n    if p.is_absolute() or ".." in p.parts or "\\\\" in name:\n        raise ValueError(f"Membro inseguro no ZIP: {name}")\n    return p\n\n\ndef parse_yolo(raw, width, height, names=NAMES, source="label"):\n    objects = []\n    for line_no, line in enumerate(raw.decode("utf-8-sig").splitlines(), 1):\n        if not line.strip():\n            continue\n        parts = line.split()\n        if len(parts) != 5:\n            raise ValueError(f"{source}:{line_no}: esperados cinco campos.")\n        vals = [float(s) for s in parts]\n        if not all(math.isfinite(v) for v in vals):\n            raise ValueError(f"{source}:{line_no}: valor não finito.")\n        c, x, y, w, h = vals\n        if int(c) != c or not 0 <= c < len(names) or min(w, h) <= 0:\n            raise ValueError(f"{source}:{line_no}: classe ou dimensão inválida.")\n        box = [x-w/2, y-h/2, x+w/2, y+h/2]\n        if min(box) < -0.002 or max(box) > 1.002:\n            raise ValueError(f"{source}:{line_no}: caixa fora da imagem.")\n        box = [max(0, min(1, v)) for v in box]\n        objects.append({"classe_id": int(c), "classe": names[int(c)],\n                        "bbox": [box[0]*width, box[1]*height,\n                                 box[2]*width, box[3]*height],\n                        "ocluido": None, "truncado": None})\n    return objects\n\n\ndef hierarchy_mapping(cfg, class_rows, hierarchy):\n    if cfg["nomes"] != NAMES or len(cfg["mids_raizes"]) != len(NAMES):\n        raise ValueError("Classes da preparação incompatíveis com o projeto.")\n    descriptions = {r[0]: r[1] for r in class_rows if len(r) >= 2}\n    roots = cfg["mids_raizes"]\n    if any(mid not in descriptions for mid in roots):\n        raise ValueError("A descrição oficial não contém todas as raízes.")\n    descendants, ancestors = defaultdict(set), defaultdict(set)\n\n    def walk(node, parents=()):\n        mid = node["LabelName"]\n        ancestors[mid].update(parents)\n        descendants[mid].add(mid)\n        for parent in parents:\n            descendants[parent].add(mid)\n        for child in node.get("Subcategory", []):\n            walk(child, parents + (mid,))\n\n    walk(hierarchy)\n    excluded_names = set(cfg.get("ambiguas_excluidas", []))\n    excluded = {mid for mid, name in descriptions.items() if name in excluded_names}\n    candidates = defaultdict(set)\n    root_desc = {}\n    for cid, mid in enumerate(roots):\n        if mid not in descendants:\n            raise ValueError(f"Raiz ausente da hierarquia: {mid}")\n        root_desc[cid] = set(descendants[mid])\n        for sub in root_desc[cid]:\n            candidates[sub].add(cid)\n    ambiguous = excluded | {mid for mid, ids in candidates.items() if len(ids) > 1}\n    mapping = {mid: next(iter(ids)) for mid, ids in candidates.items()\n               if mid not in ambiguous}\n    return {"roots": roots, "descendants": root_desc, "ancestors": ancestors,\n            "mapping": mapping, "ambiguous": ambiguous,\n            "descriptions": descriptions,\n            "relevant_labels": set().union(*root_desc.values(),\n                                            *(ancestors[r] for r in roots))}\n\n\ndef prepare_oi_annotations(box_rows, label_rows, info):\n    """Cobertura conservadora: positivo na raiz ou negativo na raiz/ancestral.\n\n    Uma positiva apenas numa subclasse não prova cobertura exaustiva da classe\n    agregada. Grupos, representações, atributos desconhecidos ou caixas inválidas\n    tornam essa classe/imagem não avaliável, em vez de criarem falsos negativos.\n    """\n    labels = defaultdict(lambda: {"positive": set(), "negative": set()})\n    for row in label_rows:\n        if row.get("Source") not in {"verification", "crowdsource-verification"}:\n            continue\n        mid, confidence = row["LabelName"], float(row["Confidence"])\n        if mid not in info["relevant_labels"]:\n            continue\n        if confidence in (0, 1):\n            key = "positive" if confidence == 1 else "negative"\n            labels[row["ImageID"]][key].add(mid)\n    boxes, blocked = defaultdict(list), defaultdict(lambda: defaultdict(set))\n    audited = Counter()\n    for row in box_rows:\n        mid, iid = row["LabelName"], row["ImageID"]\n        affected = [cid for cid, mids in info["descendants"].items() if mid in mids]\n        if not affected:\n            continue\n        if mid in info["ambiguous"]:\n            for cid in affected:\n                blocked[iid][cid].add("subclasse_ambigua_excluida")\n            audited["caixas_subclasse_ambigua"] += 1\n            continue\n        cid = info["mapping"].get(mid)\n        if cid is None:\n            continue\n        flags = {k: int(float(row.get(k, -1))) for k in\n                 ["IsGroupOf", "IsDepiction", "IsOccluded", "IsTruncated"]}\n        reasons = []\n        if row.get("Source") not in {"xclick", "activemil"}:\n            reasons.append("origem_caixa_nao_reconhecida")\n        if flags["IsGroupOf"] != 0:\n            reasons.append("grupo_ou_atributo_desconhecido")\n        if flags["IsDepiction"] != 0:\n            reasons.append("representacao_ou_atributo_desconhecido")\n        xyxy = [float(row[k]) for k in ["XMin", "YMin", "XMax", "YMax"]]\n        if (not all(math.isfinite(v) for v in xyxy) or min(xyxy) < 0\n                or max(xyxy) > 1 or xyxy[2] <= xyxy[0] or xyxy[3] <= xyxy[1]):\n            reasons.append("coordenadas_invalidas")\n        if reasons:\n            blocked[iid][cid].update(reasons)\n            audited.update(reasons)\n            continue\n        boxes[iid].append({"classe_id": cid, "classe": NAMES[cid],\n                           "bbox_norm": xyxy, "mid_original": mid,\n                           "ocluido": flags["IsOccluded"] == 1 if flags["IsOccluded"] >= 0 else None,\n                           "truncado": flags["IsTruncated"] == 1 if flags["IsTruncated"] >= 0 else None})\n    records = {}\n    for iid in sorted(set(labels) | set(boxes)):\n        gt = []\n        seen = set()\n        for obj in boxes[iid]:\n            key = (obj["classe_id"], *(round(x, 8) for x in obj["bbox_norm"]))\n            if key not in seen:\n                seen.add(key)\n                gt.append(obj)\n        known, positive, negative, reasons = 0, 0, 0, {}\n        present = {o["classe_id"] for o in gt}\n        for cid, root in enumerate(info["roots"]):\n            pos = root in labels[iid]["positive"]\n            neg = bool(({root} | info["ancestors"][root]) & labels[iid]["negative"])\n            has_positive_label = bool(info["descendants"][cid] & labels[iid]["positive"])\n            reason = set(blocked[iid].get(cid, set()))\n            if neg and (cid in present or has_positive_label):\n                reason.add("contradicao_positivo_negativo")\n            if pos and cid not in present:\n                reason.add("positivo_sem_caixa_utilizavel")\n            if reason:\n                reasons[str(cid)] = sorted(reason)\n                continue\n            if neg:\n                known |= 1 << cid\n                negative |= 1 << cid\n            elif pos and cid in present:\n                known |= 1 << cid\n                positive |= 1 << cid\n        if positive or known & ALERT_MASK == ALERT_MASK:\n            records[iid] = {"id": iid, "source_split": "validation",\n                            "known_mask": known, "positive_mask": positive,\n                            "negative_mask": negative, "gt_normalizado": gt,\n                            "classes_positivas_avaliaveis": sorted(c for c in range(12)\n                                                                 if positive & (1 << c)),\n                            "restricoes_cobertura": reasons}\n    return records, dict(audited)\n\n\ndef iou_matrix(gt, pred):\n    if not gt or not pred:\n        return np.zeros((len(gt), len(pred)))\n    a = np.asarray([o["bbox"] for o in gt], dtype=float)\n    b = np.asarray([o["bbox"] for o in pred], dtype=float)\n    lt = np.maximum(a[:, None, :2], b[None, :, :2])\n    rb = np.minimum(a[:, None, 2:], b[None, :, 2:])\n    intersection = np.maximum(rb-lt, 0).prod(axis=2)\n    aa = np.maximum(a[:, 2:]-a[:, :2], 0).prod(axis=1)\n    ba = np.maximum(b[:, 2:]-b[:, :2], 0).prod(axis=1)\n    return intersection / np.maximum(aa[:, None] + ba[None, :] - intersection, 1e-12)\n\n\ndef ap_matches(gt, pred, thresholds=np.linspace(.5, .95, 10)):\n    correct = np.zeros((len(pred), len(thresholds)), dtype=bool)\n    matrix = iou_matrix(gt, pred)\n    order = sorted(range(len(pred)), key=lambda pi: (-pred[pi]["confianca"], pi))\n    for ti, threshold in enumerate(thresholds):\n        used = set()\n        for pi in order:\n            options = [gi for gi, g in enumerate(gt) if gi not in used\n                       and g["classe_id"] == pred[pi]["classe_id"]\n                       and matrix[gi, pi] >= threshold]\n            if options:\n                gi = max(options, key=lambda k: (matrix[k, pi], -k))\n                used.add(gi)\n                correct[pi, ti] = True\n    return correct\n\n\ndef ratio(n, d):\n    return n/d if d else None\n\n\ndef alert_metrics(tp, fn, fp):\n    return {"tp": int(tp), "fn": int(fn), "fp": int(fp),\n            "precision": ratio(tp, tp+fp), "recall": ratio(tp, tp+fn),\n            "f1": ratio(2*tp, 2*tp+fp+fn)}\n\n\ndef alert_on(obj):\n    return bool(obj and LEVEL[obj["prioridade"]] >= LEVEL["medio"])\n\n\ndef with_policy(objects, width, height, policy):\n    return [{**o, **policy.avaliar_bbox(o["classe"], o["bbox"], width, height,\n                                       versao="v2")} for o in objects]\n\n\ndef record_event(record, model_name, event, g=None, p=None, iou=None, all_pred=()):\n    gbox = g["bbox"] if g else None\n    pbox = p["bbox"] if p else None\n    area = ((gbox[2]-gbox[0])*(gbox[3]-gbox[1]) /\n            (record["width"]*record["height"])) if gbox else None\n    side_at_640 = (min(gbox[2]-gbox[0], gbox[3]-gbox[1]) *\n                   640/max(record["width"], record["height"])) if gbox else None\n    flags = []\n    near_best = None\n    if event == "objeto_nao_detectado":\n        if side_at_640 < 32:\n            flags.append("objeto_pequeno_na_entrada_640")\n        if g.get("ocluido") is True:\n            flags.append("oclusao_anotada")\n        if g.get("truncado") is True:\n            flags.append("truncamento_anotado")\n        overlaps = iou_matrix([g], all_pred)[0] if all_pred else []\n        same = [i for i, o in enumerate(all_pred) if o["classe_id"] == g["classe_id"]]\n        if same:\n            best = max(same, key=lambda i: overlaps[i])\n            near_best = float(overlaps[best])\n            if overlaps[best] >= .5 and all_pred[best]["confianca"] < .25:\n                flags.append("deteccao_abaixo_da_confianca_025")\n            elif .1 <= overlaps[best] < .5:\n                flags.append("caixa_mesma_classe_iou_010_a_050")\n        if any(overlaps[i] >= .5 and o["classe_id"] != g["classe_id"]\n               and o["confianca"] >= .25 for i, o in enumerate(all_pred)):\n            flags.append("sobreposicao_com_previsao_de_outra_classe")\n        if not flags:\n            flags.append("omissao_sem_indicio_automatico")\n    if event == "previsao_nao_associada":\n        flags.append("previsao_sem_correspondencia_na_classe_avaliavel")\n    return {"coorte": record["coorte"], "imagem": record["name"], "id": record["id"],\n            "modelo": model_name, "evento": event,\n            "classe_id": g["classe_id"] if g else p["classe_id"],\n            "classe": g["classe"] if g else p["classe"],\n            "bbox_ref": json.dumps(gbox) if g else "",\n            "bbox_pred": json.dumps(pbox) if p else "",\n            "iou": iou, "confianca": p["confianca"] if p else None,\n            "prioridade_ref": g["prioridade"] if g else "sem_prioridade",\n            "prioridade_pred": p["prioridade"] if p else "sem_prioridade",\n            "ha_alerta_ref": alert_on(g), "ha_alerta_pred": alert_on(p),\n            "area_relativa_gt": area, "menor_lado_gt_em_640": side_at_640,\n            "oclusao_anotada": g.get("ocluido") if g else None,\n            "truncamento_anotado": g.get("truncado") if g else None,\n            "melhor_iou_mesma_classe": near_best,\n            "indicios": "|".join(flags)}\n\n\ndef analyse_prediction(record, pred_low, model_name, policy, matcher):\n    if record.get("role") not in {"home_val", "val_nova"}:\n        raise ValueError("Somente validação pode entrar na inferência do notebook 11.")\n    known = int(record["known_mask"])\n    gt = with_policy([g for g in record["gt"] if known & (1 << g["classe_id"])],\n                     record["width"], record["height"], policy)\n    pred_low_known = [p for p in pred_low if known & (1 << p["classe_id"])]\n    pred_all = with_policy([p for p in pred_low if p["confianca"] >= .25],\n                           record["width"], record["height"], policy)\n    pred = [p for p in pred_all if known & (1 << p["classe_id"])]\n    pairs, missing, extras = matcher(gt, pred)\n    events = []\n    for gi, pi, iou in pairs:\n        events.append(record_event(record, model_name, "correspondencia",\n                                   gt[gi], pred[pi], float(iou)))\n    for gi in missing:\n        events.append(record_event(record, model_name, "objeto_nao_detectado",\n                                   g=gt[gi], all_pred=pred_low_known))\n    for pi in extras:\n        events.append(record_event(record, model_name, "previsao_nao_associada", p=pred[pi]))\n    tp = sum(e["ha_alerta_ref"] and e["ha_alerta_pred"] for e in events)\n    fn = sum(e["ha_alerta_ref"] and not e["ha_alerta_pred"] for e in events)\n    fp = sum(not e["ha_alerta_ref"] and e["ha_alerta_pred"] for e in events)\n    high_n = sum(g["prioridade"] == "alto" for g in gt)\n    high_missing = sum(e["prioridade_ref"] == "alto" and not e["ha_alerta_pred"] for e in events)\n    messages = [policy.gerar_mensagem(p, "medio") for p in pred_all if alert_on(p)]\n    if any(not isinstance(m, str) or not m.strip() for m in messages):\n        raise RuntimeError("A política produziu alerta sem texto.")\n    image = {"coorte": record["coorte"], "imagem": record["name"], "id": record["id"],\n             "modelo": model_name, "tp_alerta": tp, "fn_alerta": fn, "fp_alerta": fp,\n             "alta_ref": high_n, "alta_sem_alerta": high_missing,\n             "tp_detector": len(pairs), "fn_detector": len(missing), "fp_detector": len(extras),\n             "gt_avaliaveis": len(gt), "gt_ignorados": len(record["gt"])-len(gt),\n             "predicoes_ignoradas_conf_025": len(pred_all)-len(pred),\n             "cobertura_completa_alertas": known & ALERT_MASK == ALERT_MASK,\n             "ha_alerta_ref": any(alert_on(g) for g in gt),\n             "ha_alerta_pred": any(alert_on(p) for p in pred),\n             "mensagem_alerta": "\\n".join(messages)}\n    stats = (ap_matches(gt, pred_low_known),\n             np.array([p["confianca"] for p in pred_low_known], dtype=float),\n             np.array([p["classe_id"] for p in pred_low_known], dtype=int),\n             np.array([g["classe_id"] for g in gt], dtype=int))\n    return image, events, stats\n\n\ndef summarise(images, events, stats, ap_function=None):\n    if not images:\n        raise ValueError("Coorte de avaliação vazia.")\n    total = lambda k: sum(int(r[k]) for r in images)\n    output = {"imagens": len(images), "conf_alertas": .25,\n              "alertas_por_objeto": alert_metrics(total("tp_alerta"), total("fn_alerta"), total("fp_alerta")),\n              "alta_ref": total("alta_ref"), "alta_sem_alerta": total("alta_sem_alerta"),\n              "taxa_alta_sem_alerta": ratio(total("alta_sem_alerta"), total("alta_ref")),\n              "detector_conf_025": alert_metrics(total("tp_detector"), total("fn_detector"), total("fp_detector")),\n              "gt_ignorados_sem_cobertura": total("gt_ignorados"),\n              "predicoes_ignoradas_conf_025": total("predicoes_ignoradas_conf_025")}\n    complete = [r for r in images if r["cobertura_completa_alertas"]]\n    output["imagens_com_cobertura_completa_alertas"] = len(complete)\n    output["alertas_por_imagem_cobertura_completa"] = alert_metrics(\n        sum(r["ha_alerta_ref"] and r["ha_alerta_pred"] for r in complete),\n        sum(r["ha_alerta_ref"] and not r["ha_alerta_pred"] for r in complete),\n        sum(not r["ha_alerta_ref"] and r["ha_alerta_pred"] for r in complete))\n    correct = np.concatenate([s[0] for s in stats], axis=0)\n    scores = np.concatenate([s[1] for s in stats])\n    pred_cls = np.concatenate([s[2] for s in stats])\n    target_cls = np.concatenate([s[3] for s in stats])\n    ap = {}\n    if ap_function is not None and len(target_cls) and len(pred_cls):\n        result = ap_function(correct, scores, pred_cls, target_cls, plot=False,\n                             names=dict(enumerate(NAMES)))\n        values, ids = np.asarray(result[5]), np.asarray(result[6], dtype=int)\n        if values.shape != (len(ids), 10):\n            raise RuntimeError("Retorno de AP incompatível com a versão fixada.")\n        ap = {int(c): a for c, a in zip(ids, values)}\n    elif ap_function is not None and len(target_cls):\n        ap = {int(c): np.zeros(10) for c in np.unique(target_cls)}\n    output["map50_recorte"] = float(np.mean([a[0] for a in ap.values()])) if ap else None\n    output["map50_95_recorte"] = float(np.mean(list(ap.values()))) if ap else None\n    class_rows = []\n    for cid, name in enumerate(NAMES):\n        part = [e for e in events if e["classe_id"] == cid]\n        a = ap.get(cid)\n        n_gt = int((target_cls == cid).sum())\n        class_rows.append({"classe_id": cid, "classe": name, "gt_avaliaveis": n_gt,\n                           "ap50": float(a[0]) if a is not None else None,\n                           "ap50_95": float(a.mean()) if a is not None else None,\n                           "suporte_reduzido": n_gt < 30,\n                           **alert_metrics(sum(e["ha_alerta_ref"] and e["ha_alerta_pred"] for e in part),\n                                           sum(e["ha_alerta_ref"] and not e["ha_alerta_pred"] for e in part),\n                                           sum(not e["ha_alerta_ref"] and e["ha_alerta_pred"] for e in part)),\n                           "alta_ref": sum(e["prioridade_ref"] == "alto" for e in part),\n                           "alta_sem_alerta": sum(e["prioridade_ref"] == "alto" and not e["ha_alerta_pred"] for e in part)})\n    return output, class_rows\n\n\ndef model_predictions(model, record, device):\n    if record["role"] not in {"home_val", "val_nova"}:\n        raise ValueError("Tentativa de inferência em reserva ou conjunto não autorizado.")\n    result = model.predict(str(record["image"]), conf=.001, imgsz=640, rect=True,\n                           iou=.70, max_det=300, device=device, half=False,\n                           verbose=False, save=False)[0]\n    if Path(result.path).resolve() != Path(record["image"]).resolve():\n        raise RuntimeError("A inferência retornou uma imagem diferente.")\n    if result.boxes is None:\n        return []\n    return [{"classe_id": int(c), "classe": NAMES[int(c)],\n             "bbox": [float(v) for v in box], "confianca": float(score)}\n            for c, box, score in zip(result.boxes.cls.cpu().numpy(),\n                                      result.boxes.xyxy.cpu().numpy(),\n                                      result.boxes.conf.cpu().numpy())]\n\n\ndef evaluate_model(model, records, model_name, output_dir, policy, matcher, device,\n                   ap_function):\n    output_dir = Path(output_dir)\n    output_dir.mkdir(parents=True, exist_ok=True)\n    images, events, stats = [], [], []\n    cache_file = output_dir / "predicoes_conf_0001.jsonl.gz"\n    with gzip.open(cache_file, "wt", encoding="utf-8") as cache:\n        for i, record in enumerate(records, 1):\n            pred = model_predictions(model, record, device)\n            cache.write(json.dumps({"id": record["id"], "imagem": record["name"],\n                                    "predicoes": pred}, ensure_ascii=False) + "\\n")\n            im, ev, st = analyse_prediction(record, pred, model_name, policy, matcher)\n            images.append(im)\n            events.extend(ev)\n            stats.append(st)\n            if i % 100 == 0 or i == len(records):\n                print(f"{record[\'coorte\']} / {model_name}: {i}/{len(records)}", flush=True)\n    summary, classes = summarise(images, events, stats, ap_function)\n    summary.update(modelo=model_name, coorte=records[0]["coorte"],\n                   protocolo="12 classes; AP por confiança; alertas por IoU; cobertura conservadora",\n                   nao_e_openimages_challenge=True, selecao_automatica_de_modelo=False)\n    save_json(output_dir / "metricas.json", summary)\n    save_csv(output_dir / "metricas_por_classe.csv", classes)\n    save_csv(output_dir / "alertas_por_imagem.csv", images)\n    save_csv(output_dir / "avaliacao_objetos.csv", events)\n    (output_dir / "alertas_texto.txt").write_text(\n        "\\n\\n".join(r["imagem"]+"\\n"+r["mensagem_alerta"] for r in images if r["mensagem_alerta"]),\n        encoding="utf-8")\n    return {"summary": summary, "classes": classes, "images": images, "events": events,\n            "cache_file": cache_file}\n', 'diag11_data.py': '"""Entradas, auditoria documental e nova validação, sem consulta a previsões."""\nfrom __future__ import annotations\n\nimport csv\nimport hashlib\nimport io\nimport json\nimport math\nimport os\nimport time\nimport urllib.request\nimport zipfile\nfrom collections import Counter, defaultdict\nfrom concurrent.futures import ThreadPoolExecutor\nfrom pathlib import Path\n\nimport numpy as np\nfrom PIL import Image\nfrom scipy.fft import dctn\n\nfrom diag11_core import (ALL_KNOWN, ALERT_MASK, NAMES, canonical, parse_yolo,\n                         safe_member, save_csv, save_json, sha256_file)\n\nSOURCE_URLS = {\n    "classes": "https://storage.googleapis.com/openimages/v7/oidv7-class-descriptions-boxable.csv",\n    "hierarquia": "https://storage.googleapis.com/openimages/2018_04/bbox_labels_600_hierarchy.json",\n    "caixas": "https://storage.googleapis.com/openimages/v5/validation-annotations-bbox.csv",\n    "rotulos": "https://storage.googleapis.com/openimages/v5/validation-annotations-human-imagelabels-boxable.csv",\n    "metadata": "https://storage.googleapis.com/openimages/2018_04/validation/validation-images-with-rotation.csv",\n}\n\n\ndef download(url, path, retries=3):\n    """Download atômico, com recibo local e validação de tamanho quando informado."""\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    receipt = path.with_name(path.name + ".download.json")\n    if path.is_file() and receipt.is_file():\n        prior = json.loads(receipt.read_text())\n        if prior.get("url") == url and prior.get("sha256") == sha256_file(path):\n            return path\n    last = None\n    for attempt in range(retries):\n        tmp = path.with_name(path.name + ".tmp")\n        try:\n            req = urllib.request.Request(url, headers={"User-Agent": "TCC-Diagnostico-11/1.0"})\n            with urllib.request.urlopen(req, timeout=45) as response, tmp.open("wb") as out:\n                expected = response.headers.get("Content-Length")\n                for block in iter(lambda: response.read(2**20), b""):\n                    out.write(block)\n            if expected and tmp.stat().st_size != int(expected):\n                raise IOError("Tamanho recebido difere do Content-Length.")\n            if not tmp.stat().st_size:\n                raise IOError("Arquivo recebido vazio.")\n            tmp.replace(path)\n            save_json(receipt, {"url": url, "sha256": sha256_file(path),\n                                "bytes": path.stat().st_size})\n            return path\n        except Exception as exc:\n            last = type(exc).__name__ + ": " + str(exc)[:180]\n            tmp.unlink(missing_ok=True)\n            if attempt + 1 < retries:\n                time.sleep(min(2**attempt, 4))\n    raise IOError(f"Download não concluído: {url}; {last}")\n\n\ndef csv_rows(path):\n    with Path(path).open(encoding="utf-8-sig", newline="") as f:\n        yield from csv.DictReader(f)\n\n\ndef load_history(csv_path, json_path, expected_hash):\n    split = json.loads(Path(json_path).read_text())\n    stated = split.pop("divisao_sha256")\n    computed = hashlib.sha256(canonical(split).encode()).hexdigest()\n    if stated != expected_hash or computed != stated:\n        raise RuntimeError("Divisão histórica não corresponde à execução 10.")\n    rows = list(csv_rows(csv_path))\n    if len(rows) != 100000 or len({r["id"] for r in rows}) != 100000:\n        raise RuntimeError("O manifesto histórico precisa de 100.000 IDs distintos.")\n    if Counter(r["split"] for r in rows) != {"train": 80000, "test": 20000}:\n        raise RuntimeError("Divisão histórica diferente de 80/20.")\n    for label, key in [("train", "ids_treino"), ("test", "ids_teste")]:\n        if {r["id"] for r in rows if r["split"] == label} != set(split[key]):\n            raise RuntimeError("CSV e JSON históricos divergem.")\n    hashes = {r["imagem_sha256"] for r in rows}\n    if len(hashes) != len(rows):\n        raise RuntimeError("O histórico contém imagens repetidas em bytes.")\n    return {"ids": {r["id"] for r in rows}, "hashes": hashes, "rows": rows,\n            "split_hash": stated}\n\n\ndef phash(path):\n    with Image.open(path) as image:\n        small = np.asarray(image.convert("L").resize((32, 32), Image.Resampling.LANCZOS),\n                           dtype=float)\n    low = dctn(small, norm="ortho")[:8, :8]\n    bits = low > np.median(low.flatten()[1:])\n    bits[0, 0] = False\n    return int.from_bytes(np.packbits(bits.flatten()).tobytes(), "big")\n\n\nclass NearIndex:\n    """pHash de 64 bits, distância <=4; sinal de semelhança, não prova semântica."""\n    def __init__(self):\n        self.tables = [defaultdict(list) for _ in range(8)]\n\n    def add(self, value, name):\n        for band in range(8):\n            self.tables[band][(value >> (8*band)) & 255].append((value, name))\n\n    def find(self, value, max_distance=4):\n        checked = set()\n        for band in range(8):\n            for prior, name in self.tables[band].get((value >> (8*band)) & 255, []):\n                if (prior, name) in checked:\n                    continue\n                checked.add((prior, name))\n                if (prior ^ value).bit_count() <= max_distance:\n                    return name\n        return None\n\n\ndef extract_home(zip_path, destination, expected_zip, expected_test, expected_policy):\n    if sha256_file(zip_path) != expected_zip:\n        raise RuntimeError("ZIP Home diferente da entrada preservada.")\n    destination = Path(destination)\n    records, hashes, near = [], set(), NearIndex()\n    with zipfile.ZipFile(zip_path) as z:\n        if len(z.namelist()) != len(set(z.namelist())):\n            raise RuntimeError("Há membros repetidos no ZIP Home.")\n        manifest = json.loads(z.read("verificacao_preparacao.json"))\n        if (manifest["test"]["sha256"] != expected_test or\n                manifest["test"]["imagens"] != 403 or\n                {int(k): v for k, v in manifest["classes"].items()} != dict(enumerate(NAMES))):\n            raise RuntimeError("Manifesto Home incompatível.")\n        root = manifest["dataset_dir"]\n        items = {r["arquivo"]: r for r in manifest["arquivos"]}\n        if len(items) != len(manifest["arquivos"]):\n            raise RuntimeError("Manifesto Home com membros repetidos.")\n        for name in z.namelist():\n            safe_member(name)\n            if name.startswith("codigo/") and name.endswith(".py"):\n                raw = z.read(name)\n                if name in items and hashlib.sha256(raw).hexdigest() != items[name]["sha256"]:\n                    raise RuntimeError("Código Home diferente do manifesto.")\n                out = destination / name\n                out.parent.mkdir(parents=True, exist_ok=True)\n                out.write_bytes(raw)\n        for name, item in sorted(items.items()):\n            is_image = (name.startswith(f"{root}/images/") and\n                        Path(name).suffix.lower() in {".jpg", ".jpeg", ".png", ".webp", ".bmp"})\n            if is_image:\n                hashes.add(item["sha256"])\n            role = None\n            if name.startswith(f"{root}/images/train/raw_train/"):\n                role = "home_train"\n            elif name.startswith(f"{root}/images/val/raw_train/"):\n                role = "home_val"\n            if not is_image or role is None:\n                continue\n            raw = z.read(name)\n            if hashlib.sha256(raw).hexdigest() != item["sha256"]:\n                raise RuntimeError(f"Imagem Home alterada: {name}")\n            image = destination / name\n            image.parent.mkdir(parents=True, exist_ok=True)\n            image.write_bytes(raw)\n            label_name = name.replace("/images/", "/labels/").rsplit(".", 1)[0] + ".txt"\n            if label_name not in items:\n                raise RuntimeError("Label Home ausente do manifesto.")\n            label_raw = z.read(label_name)\n            if hashlib.sha256(label_raw).hexdigest() != items[label_name]["sha256"]:\n                raise RuntimeError("Label Home alterado.")\n            with Image.open(image) as im:\n                width, height = im.size\n                im.verify()\n            gt = parse_yolo(label_raw, width, height, source=label_name)\n            record = {"id": "home:" + name, "name": name.split("/raw_train/")[-1],\n                      "image": image, "role": role, "coorte": "home_val_historica",\n                      "width": width, "height": height, "known_mask": ALL_KNOWN,\n                      "sha256": item["sha256"], "label_sha256": items[label_name]["sha256"],\n                      "gt": gt}\n            record["phash"] = phash(image)\n            near.add(record["phash"], name)\n            records.append(record)\n    counts = Counter(r["role"] for r in records)\n    if counts != {"home_train": 1824, "home_val": 456}:\n        raise RuntimeError(f"Home inesperado: {dict(counts)}")\n    train_hash = {r["sha256"] for r in records if r["role"] == "home_train"}\n    val_hash = {r["sha256"] for r in records if r["role"] == "home_val"}\n    if train_hash & val_hash:\n        raise RuntimeError("Home TRAIN e VAL atuais compartilham imagens em bytes.")\n    policy_file = destination / "codigo/src/politica_alertas.py"\n    if not policy_file.is_file() or sha256_file(policy_file) != expected_policy:\n        raise RuntimeError("Política V2 diferente da versão preservada.")\n    audit = {"home_train_atual": 1824, "home_val_atual": 456,\n             "test_home_preservado": 403, "test_extraido": False,\n             "intersecao_bytes_home_train_val_atual": 0,\n             "divergencia_documental_1822_1824": "Não reconciliada; o pacote atual tem 1824.",\n             "separacao_atual_nao_comprova_historico_baseline": True}\n    return records, hashes, near, audit\n\n\ndef audit_baseline_metadata(checkpoint, expected_hash, home_records, evidence=None):\n    if sha256_file(checkpoint) != expected_hash:\n        raise RuntimeError("Peso baseline diferente da referência.")\n    # Só o checkpoint autenticado pelo SHA esperado é desserializado.\n    import torch\n    obj = torch.load(checkpoint, map_location="cpu", weights_only=False)\n    args = obj.get("train_args", {})\n    fields = ["data", "epochs", "batch", "imgsz", "seed", "optimizer", "lr0",\n              "project", "name", "fraction"]\n    audit = {"modelo_sha256": expected_hash, "data_checkpoint": str(obj.get("date")),\n             "train_args_preservados": {k: args[k] for k in fields if k in args},\n             "estado_proveniencia_original": "nao_determinado",\n             "home_val_usada_no_treino_original": None,\n             "motivo": "Caminhos e argumentos não contêm a lista de imagens vistas no treino original."}\n    del obj\n    if evidence is not None:\n        required = {"modelo_sha256", "train_image_sha256", "val_image_sha256"}\n        if not required <= evidence.keys() or evidence["modelo_sha256"] != expected_hash:\n            audit["evidencia_original"] = "incompativel_ou_sem_vinculo_ao_peso"\n        else:\n            train = set(evidence["train_image_sha256"])\n            val = set(evidence["val_image_sha256"])\n            valid_sha = lambda x: isinstance(x, str) and len(x) == 64 and all(c in "0123456789abcdef" for c in x)\n            if not train or any(not valid_sha(h) for h in train | val):\n                raise ValueError("Manifesto original precisa de hashes SHA-256 válidos.")\n            current = {r["sha256"] for r in home_records if r["role"] == "home_val"}\n            overlap = train & current\n            complete = evidence.get("manifesto_treino_completo") is True\n            audit.update(estado_proveniencia_original="manifesto_documental_analisado",\n                         evidencia_original="documento_vinculado_ao_sha_do_modelo",\n                         home_val_usada_no_treino_original=True if overlap else (False if complete else None),\n                         manifesto_declarado_completo=complete,\n                         imagens_home_val_no_manifesto_train_original=len(overlap),\n                         imagens_home_val_no_manifesto_val_original=len(current & val),\n                         motivo="Conclusão limitada à completude e correção do manifesto apresentado.")\n    return audit\n\n\ndef stable_order(iid, seed=42):\n    return hashlib.sha256(f"{seed}|{iid}".encode()).hexdigest(), iid\n\n\ndef balanced_ids(records, n, seed=42, per_class=45):\n    """Diversidade por classe e preenchimento determinístico, sem usar previsões."""\n    if len(records) < n:\n        raise ValueError(f"Há somente {len(records)} imagens elegíveis; exigidas {n}.")\n    groups = {c: sorted((iid for iid, r in records.items() if\n                         r["positive_mask"] & (1 << c)), key=lambda x: stable_order(x, seed))\n              for c in range(12)}\n    order = sorted(groups, key=lambda c: (len(groups[c]), c))\n    chosen, counts = set(), Counter()\n    # Diversificar exemplos negativos quando as seis classes têm ausência verificada.\n    negatives = sorted((iid for iid, r in records.items() if\n                        r["known_mask"] & ALERT_MASK == ALERT_MASK and\n                        not r["positive_mask"] & ALERT_MASK), key=lambda x: stable_order(x, seed))\n    for iid in negatives[:min(len(negatives), n//10)]:\n        chosen.add(iid)\n        counts.update(c for c in range(12) if records[iid]["positive_mask"] & (1 << c))\n    for target in range(1, per_class+1):\n        for cid in order:\n            if len(chosen) >= n:\n                break\n            if counts[cid] >= target:\n                continue\n            for iid in groups[cid]:\n                if iid not in chosen:\n                    chosen.add(iid)\n                    counts.update(c for c in range(12) if records[iid]["positive_mask"] & (1 << c))\n                    break\n    for iid in sorted(records, key=lambda x: stable_order(x, seed)):\n        if len(chosen) == n:\n            break\n        chosen.add(iid)\n    return sorted(chosen, key=lambda x: stable_order(x, seed))\n\n\ndef split_pool(records, n_reserve, seed=42):\n    """Estratos pela classe positiva mais rara; a reserva não entra na inferência."""\n    frequencies = Counter(c for r in records.values() for c in range(12)\n                          if r["positive_mask"] & (1 << c))\n    groups = defaultdict(list)\n    for iid, record in records.items():\n        classes = record["classes_positivas_avaliaveis"]\n        primary = min(classes, key=lambda c: (frequencies[c], c)) if classes else -1\n        groups[primary].append(iid)\n    total = len(records)\n    if not 0 < n_reserve < total:\n        raise ValueError("A reserva precisa ser menor que o conjunto novo.")\n    quotas = {c: len(ids)*n_reserve//total for c, ids in groups.items()}\n    order = sorted(groups, key=lambda c: (-(len(groups[c])*n_reserve % total), c))\n    for cid in order[:n_reserve-sum(quotas.values())]:\n        quotas[cid] += 1\n    selected = set()\n    for cid, ids in groups.items():\n        selected.update(sorted(ids, key=lambda x: stable_order(x, seed+1))[:quotas[cid]])\n    if len(selected) != n_reserve:\n        raise RuntimeError("A divisão não produziu a reserva planejada.")\n    return {iid: ("reserva_futura" if iid in selected else "val_nova") for iid in records}\n\n\ndef filter_metadata(records, metadata_path, prior_ids):\n    metadata, exclusions = {}, Counter()\n    for row in csv_rows(metadata_path):\n        iid = row["ImageID"]\n        if iid not in records:\n            continue\n        if row["Subset"] != "validation":\n            raise ValueError("A metadata contém uma origem diferente de validation.")\n        if iid in prior_ids:\n            exclusions["id_presente_nos_100k"] += 1\n            continue\n        try:\n            rotation = float(row["Rotation"])\n        except (TypeError, ValueError):\n            rotation = float("nan")\n        # Evita suposições sobre transformação das caixas e orientação do JPEG.\n        if rotation != 0:\n            exclusions["rotacao_nao_zero_ou_desconhecida"] += 1\n            continue\n        metadata[iid] = {k: row.get(k, "") for k in\n                         ["ImageID", "OriginalLandingURL", "License", "Author", "Rotation"]}\n    result = {iid: {**r, "creditos": metadata[iid]} for iid, r in records.items()\n              if iid in metadata}\n    return result, dict(exclusions)\n\n\ndef download_images(candidates, ids, directory, workers=8):\n    directory = Path(directory)\n    directory.mkdir(parents=True, exist_ok=True)\n\n    def one(iid):\n        record = dict(candidates[iid])\n        path = directory / f"oi_val_{iid}.jpg"\n        try:\n            download(f"https://open-images-dataset.s3.amazonaws.com/validation/{iid}.jpg", path)\n            with Image.open(path) as image:\n                image.verify()\n            with Image.open(path) as image:\n                width, height = image.size\n                orientation = image.getexif().get(274, 1)\n                if width < 32 or height < 32 or orientation != 1:\n                    raise ValueError("JPEG pequeno ou com orientação EXIF não trivial.")\n                image.load()\n            tiny = {g["classe_id"] for g in record["gt_normalizado"] if\n                    (g["bbox_norm"][2]-g["bbox_norm"][0])*width < 1 or\n                    (g["bbox_norm"][3]-g["bbox_norm"][1])*height < 1}\n            # A avaliação da ausência fica desconhecida para a classe afetada.\n            record["restricoes_cobertura"] = dict(record["restricoes_cobertura"])\n            for cid in tiny:\n                for key in ["known_mask", "positive_mask", "negative_mask"]:\n                    record[key] &= ~(1 << cid)\n                record["restricoes_cobertura"][str(cid)] = ["caixa_menor_que_um_pixel"]\n            record["classes_positivas_avaliaveis"] = [c for c in range(12)\n                                                      if record["positive_mask"] & (1 << c)]\n            if not (record["positive_mask"] or record["known_mask"] & ALERT_MASK == ALERT_MASK):\n                raise ValueError("Nenhuma cobertura útil após conferir as dimensões reais.")\n            record.update(image=path, width=width, height=height, sha256=sha256_file(path),\n                          phash=phash(path), name=path.name)\n            return record, None\n        except Exception as exc:\n            return None, {"id": iid, "motivo": type(exc).__name__, "detalhe": str(exc)[:240]}\n\n    valid, failures = {}, []\n    with ThreadPoolExecutor(max_workers=workers) as executor:\n        for done, (record, failure) in enumerate(executor.map(one, ids), 1):\n            if record:\n                valid[record["id"]] = record\n            else:\n                failures.append(failure)\n            if done % 100 == 0 or done == len(ids):\n                print(f"Novas imagens: {done}/{len(ids)}; válidas: {len(valid)}", flush=True)\n    return valid, failures\n\n\ndef deduplicate(records, prior_hashes, home_near, seed=42):\n    accepted, excluded, seen_hash, near = {}, [], set(), NearIndex()\n    for iid in sorted(records, key=lambda x: stable_order(x, seed)):\n        record = records[iid]\n        reason, similar = None, None\n        if record["sha256"] in prior_hashes:\n            reason = "bytes_presentes_nos_dados_anteriores"\n        elif record["sha256"] in seen_hash:\n            reason = "bytes_repetidos_no_conjunto_novo"\n        elif (similar := home_near.find(record["phash"])) is not None:\n            reason = "semelhanca_phash_com_home_train_ou_val"\n        elif (similar := near.find(record["phash"])) is not None:\n            reason = "semelhanca_phash_dentro_do_conjunto_novo"\n        if reason:\n            excluded.append({"id": iid, "motivo": reason, "semelhante_a": similar})\n            continue\n        accepted[iid] = record\n        seen_hash.add(record["sha256"])\n        near.add(record["phash"], iid)\n    return accepted, excluded\n\n\ndef freeze_dataset(records, roles, directory, source_hashes, prior_split_hash, seed=42):\n    directory = Path(directory)\n    directory.mkdir(parents=True, exist_ok=True)\n    manifest, runtime = [], []\n    for iid in sorted(roles):\n        r, role = records[iid], roles[iid]\n        boxes = []\n        for g in r["gt_normalizado"]:\n            x1, y1, x2, y2 = g["bbox_norm"]\n            boxes.append({**g, "bbox": [x1*r["width"], y1*r["height"],\n                                       x2*r["width"], y2*r["height"]]})\n        coverage = {k: r[k] for k in ["known_mask", "positive_mask", "negative_mask",\n                                      "restricoes_cobertura"]}\n        label = directory / "labels" / role / f"oi_val_{iid}.txt"\n        coverage_file = directory / "coverage" / role / f"oi_val_{iid}.json"\n        label.parent.mkdir(parents=True, exist_ok=True)\n        lines = []\n        for g in r["gt_normalizado"]:\n            x1, y1, x2, y2 = g["bbox_norm"]\n            lines.append(f"{g[\'classe_id\']} {(x1+x2)/2:.10f} {(y1+y2)/2:.10f} {(x2-x1):.10f} {(y2-y1):.10f}")\n        label.write_text("\\n".join(lines)+("\\n" if lines else ""), encoding="utf-8")\n        save_json(coverage_file, coverage)\n        image_rel = f"images/{role}/oi_val_{iid}.jpg"\n        item = {"id": iid, "role": role, "image": image_rel,\n                "label": label.relative_to(directory).as_posix(),\n                "coverage": coverage_file.relative_to(directory).as_posix(),\n                "sha256": r["sha256"], "label_sha256": sha256_file(label),\n                "coverage_sha256": sha256_file(coverage_file),\n                "width": r["width"], "height": r["height"], "known_mask": r["known_mask"],\n                "positive_mask": r["positive_mask"], "negative_mask": r["negative_mask"],\n                "phash": f"{r[\'phash\']:016x}", "creditos": r["creditos"],\n                "gt_normalizado": r["gt_normalizado"]}\n        manifest.append(item)\n        runtime.append({**r, "role": role, "coorte": "openimages_val_nova",\n                        "gt": boxes, "label": label, "coverage": coverage_file})\n    metadata = {"versao": "11-validacao-independente-v1", "seed": seed, "nomes": NAMES,\n                "fontes_metadata_sha256": source_hashes, "origem_split": "validation",\n                "divisao_100k_anterior_sha256": prior_split_hash,\n                "contagens": dict(Counter(roles.values())),\n                "criterio": "Diversidade de classes, hash determinístico e pHash; sem previsões",\n                "cobertura": "Positivo explícito na raiz ou negativo na raiz/ancestral; classes problemáticas excluídas",\n                "independencia_limitada_ao_historico_conhecido": True,\n                "pretreinamentos_publicos_nao_auditados": True,\n                "reserva_futura_avaliada": False, "arquivos": manifest}\n    dataset_hash = hashlib.sha256(canonical(metadata).encode()).hexdigest()\n    metadata["dataset_sha256"] = dataset_hash\n    save_json(directory / "manifesto_validacao_nova.json", metadata)\n    save_csv(directory / "divisao.csv", [{k: r[k] for k in\n                                        ["id", "role", "sha256", "known_mask", "positive_mask"]}\n                                       for r in manifest])\n    archive = directory.parent / f"validacao_independente_11_{dataset_hash[:12]}.zip"\n    # Os JPEGs permanecem sem recompressão.\n    with zipfile.ZipFile(archive, "w", compression=zipfile.ZIP_STORED) as z:\n        z.write(directory / "manifesto_validacao_nova.json", "manifesto_validacao_nova.json")\n        z.write(directory / "divisao.csv", "divisao.csv")\n        for r, item in zip(sorted(runtime, key=lambda r: r["id"]), manifest):\n            z.write(r["image"], item["image"])\n            z.write(r["label"], item["label"])\n            z.write(r["coverage"], item["coverage"])\n    return runtime, metadata, archive\n\n\ndef support_rows(records):\n    rows = []\n    for role in sorted({r["role"] for r in records}):\n        group = [r for r in records if r["role"] == role]\n        for cid, name in enumerate(NAMES):\n            gt = [g for r in group for g in r["gt"] if\n                  g["classe_id"] == cid and r["known_mask"] & (1 << cid)]\n            rows.append({"conjunto": role, "classe_id": cid, "classe": name,\n                         "imagens_cobertura_conhecida": sum(bool(r["known_mask"] & (1 << cid)) for r in group),\n                         "imagens_positivas_avaliaveis": sum(bool(r["positive_mask"] & (1 << cid)) for r in group),\n                         "caixas_avaliaveis": len(gt), "suporte_reduzido": len(gt) < 30})\n    return rows\n', 'diag11_reports.py': '"""Comparação pareada, indícios dos erros e galeria de inspeção."""\nfrom __future__ import annotations\n\nimport gzip\nimport html\nimport json\nimport zipfile\nfrom collections import Counter\nfrom pathlib import Path\n\nimport numpy as np\nfrom PIL import Image, ImageDraw, ImageFont\n\nfrom diag11_core import (NAMES, LEVEL, alert_on, canonical, ratio, save_csv,\n                         save_json, with_policy)\nfrom diag11_data import stable_order\n\n\ndef paired_bootstrap(old_images, new_images, repetitions=1000, seed=42):\n    old = {r["id"]: r for r in old_images}\n    new = {r["id"]: r for r in new_images}\n    if set(old) != set(new) or len(old) != len(old_images) or len(new) != len(new_images):\n        raise ValueError("Bootstrap exige as mesmas imagens, sem repetições.")\n    ids = sorted(old)\n    fields = ["tp_alerta", "fn_alerta", "fp_alerta", "alta_sem_alerta", "alta_ref"]\n    a = np.array([[old[i][k] for k in fields] for i in ids], dtype=float)\n    b = np.array([[new[i][k] for k in fields] for i in ids], dtype=float)\n\n    def scores(x):\n        tp, fn, fp, missing, high = x\n        return [100*ratio(tp, tp+fp) if tp+fp else np.nan,\n                100*ratio(tp, tp+fn) if tp+fn else np.nan,\n                100*ratio(2*tp, 2*tp+fp+fn) if 2*tp+fp+fn else np.nan,\n                fn, fp, missing, 100*ratio(missing, high) if high else np.nan]\n\n    point = np.array(scores(b.sum(axis=0))) - np.array(scores(a.sum(axis=0)))\n    rng = np.random.default_rng(seed)\n    results = []\n    for _ in range(repetitions):\n        index = rng.integers(0, len(ids), len(ids))\n        results.append(np.array(scores(b[index].sum(axis=0))) -\n                       np.array(scores(a[index].sum(axis=0))))\n    samples = np.asarray(results)\n    labels = ["precision_alertas_pp", "recall_alertas_pp", "f1_alertas_pp",\n              "fn_alertas", "fp_alertas", "alta_sem_alerta", "taxa_alta_sem_alerta_pp"]\n    rows = []\n    for j, name in enumerate(labels):\n        values = samples[np.isfinite(samples[:, j]), j]\n        low, high = np.quantile(values, [.025, .975]) if len(values) else (None, None)\n        rows.append({"metrica": name, "delta_candidato_menos_antigo": float(point[j]) if np.isfinite(point[j]) else None,\n                     "ic95_inferior": low, "ic95_superior": high,\n                     "reamostragens_validas": len(values), "imagens": len(ids),\n                     "metodo": "Bootstrap pareado por imagem; não corrige possível exposição prévia"})\n    return rows\n\n\ndef comparison_rows(evaluations):\n    rows = []\n    for cohort, models in evaluations.items():\n        for name, result in models.items():\n            s, a = result["summary"], result["summary"]["alertas_por_objeto"]\n            rows.append({"coorte": cohort, "modelo": name, "imagens": s["imagens"],\n                         "precision_alertas": a["precision"], "recall_alertas": a["recall"],\n                         "f1_alertas": a["f1"], "tp_alerta": a["tp"], "fn_alerta": a["fn"],\n                         "fp_alerta": a["fp"], "alta_ref": s["alta_ref"],\n                         "alta_sem_alerta": s["alta_sem_alerta"],\n                         "map50_recorte": s["map50_recorte"],\n                         "map50_95_recorte": s["map50_95_recorte"]})\n    return rows\n\n\ndef diagnosis_rows(evaluations):\n    rows = []\n    for cohort, models in evaluations.items():\n        for model, result in models.items():\n            events = result["events"]\n            for cid, name in enumerate(NAMES):\n                group = [e for e in events if e["classe_id"] == cid]\n                for indication in sorted({x for e in group for x in e["indicios"].split("|") if x}):\n                    part = [e for e in group if indication in e["indicios"].split("|")]\n                    rows.append({"coorte": cohort, "modelo": model, "classe": name,\n                                 "indicio": indication, "eventos": len(part),\n                                 "fn_alerta": sum(e["ha_alerta_ref"] and not e["ha_alerta_pred"] for e in part),\n                                 "alta_sem_alerta": sum(e["prioridade_ref"] == "alto" and not e["ha_alerta_pred"] for e in part),\n                                 "observacao": "Indícios podem se sobrepor; não estabelecem a causa do erro."})\n    return rows\n\n\ndef load_prediction_cache(path):\n    records = {}\n    with gzip.open(path, "rt", encoding="utf-8") as f:\n        for line in f:\n            r = json.loads(line)\n            if r["id"] in records:\n                raise ValueError("Cache tem imagem repetida.")\n            records[r["id"]] = r["predicoes"]\n    return records\n\n\ndef select_gallery_cases(old, new, limit=24, seed=42):\n    a, b = {r["id"]: r for r in old}, {r["id"]: r for r in new}\n    if set(a) != set(b):\n        raise ValueError("Galeria exige os dois modelos nas mesmas imagens.")\n    chosen = {}\n\n    def take(label, ordered, number):\n        used = 0\n        for iid in ordered:\n            if iid not in chosen and len(chosen) < limit:\n                chosen[iid] = label\n                used += 1\n                if used == number:\n                    break\n\n    worse = sorted(a, key=lambda i: (-(b[i]["alta_sem_alerta"]-a[i]["alta_sem_alerta"]),\n                                     -(b[i]["fn_alerta"]-a[i]["fn_alerta"]), stable_order(i, seed)))\n    better = sorted(a, key=lambda i: (b[i]["alta_sem_alerta"]-a[i]["alta_sem_alerta"],\n                                      b[i]["fn_alerta"]-a[i]["fn_alerta"], stable_order(i, seed)))\n    false = sorted(a, key=lambda i: (-(a[i]["fp_alerta"]+b[i]["fp_alerta"]), stable_order(i, seed)))\n    take("Candidato com mais omissões", [i for i in worse if\n         (b[i]["alta_sem_alerta"], b[i]["fn_alerta"]) >\n         (a[i]["alta_sem_alerta"], a[i]["fn_alerta"])], limit//4)\n    take("Candidato com menos omissões", [i for i in better if\n         (b[i]["alta_sem_alerta"], b[i]["fn_alerta"]) <\n         (a[i]["alta_sem_alerta"], a[i]["fn_alerta"])], limit//4)\n    take("Falsos alertas avaliáveis", [i for i in false if a[i]["fp_alerta"]+b[i]["fp_alerta"]], limit//4)\n    take("Amostra por hash, independente do erro", sorted(a, key=lambda i: stable_order(i, seed)), limit)\n    return [{"id": i, "categoria": label} for i, label in chosen.items()]\n\n\ndef font(size):\n    candidates = ["/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf",\n                  "/usr/share/fonts/truetype/liberation2/LiberationSans-Regular.ttf"]\n    for path in candidates:\n        if Path(path).is_file():\n            return ImageFont.truetype(path, size)\n    return ImageFont.load_default()\n\n\ndef draw_panel(record, objects, title):\n    canvas = Image.new("RGB", (540, 470), "#101a28")\n    draw = ImageDraw.Draw(canvas)\n    draw.text((12, 10), title, fill="#f5f7fa", font=font(18))\n    with Image.open(record["image"]) as image:\n        image = image.convert("RGB")\n        scale = min(520/image.width, 390/image.height)\n        image = image.resize((max(1, round(image.width*scale)),\n                              max(1, round(image.height*scale))), Image.Resampling.LANCZOS)\n    x0, y0 = (540-image.width)//2, 46+(390-image.height)//2\n    canvas.paste(image, (x0, y0))\n    sx, sy = image.width/record["width"], image.height/record["height"]\n    colours = {"alto": "#ff7b8a", "medio": "#ffde70",\n               "baixo": "#6ee7b7", "sem_prioridade": "#adbac7"}\n    for obj in objects:\n        cid = obj["classe_id"]\n        known = bool(record["known_mask"] & (1 << cid))\n        colour = colours[obj["prioridade"]] if known else "#a9afba"\n        x1, y1, x2, y2 = obj["bbox"]\n        box = [x0+x1*sx, y0+y1*sy, x0+x2*sx, y0+y2*sy]\n        draw.rectangle(box, outline=colour, width=2)\n        label = obj["classe"]\n        if "confianca" in obj:\n            label += f" {obj[\'confianca\']:.2f}"\n        label += " / " + obj["prioridade"] if known else " / sem cobertura"\n        # Limita o rótulo à largura visível; não altera a caixa nem as métricas.\n        tx, ty = max(0, min(box[0], 535)), max(44, box[1]-15)\n        text_box = draw.textbbox((tx, ty), label, font=font(11))\n        draw.rectangle(text_box, fill="#101a28")\n        draw.text((tx, ty), label, fill=colour, font=font(11))\n    draw.text((12, 445), "Rosa: alta | amarelo: média | verde: baixa | cinza: sem referência",\n              fill="#c6ced9", font=font(10))\n    return canvas\n\n\ndef render_case(record, old_pred, new_pred, policy, destination):\n    gt = with_policy([g for g in record["gt"] if record["known_mask"] & (1 << g["classe_id"])],\n                     record["width"], record["height"], policy)\n    old = with_policy([p for p in old_pred if p["confianca"] >= .25],\n                      record["width"], record["height"], policy)\n    new = with_policy([p for p in new_pred if p["confianca"] >= .25],\n                      record["width"], record["height"], policy)\n    panels = [draw_panel(record, gt, "Referência V2 avaliável"),\n              draw_panel(record, old, "Modelo antigo / confiança 0,25"),\n              draw_panel(record, new, "Candidato / confiança 0,25")]\n    result = Image.new("RGB", (1620, 470), "#101a28")\n    for i, panel in enumerate(panels):\n        result.paste(panel, (540*i, 0))\n    destination = Path(destination)\n    destination.parent.mkdir(parents=True, exist_ok=True)\n    result.save(destination, quality=88)\n    return destination\n\n\ndef format_percent(value):\n    return "sem suporte" if value is None else f"{100*value:.2f}%"\n\n\ndef build_report(output_dir, evaluations, cohorts, audit, bootstrap, policy, dataset_manifest, seed=42):\n    output_dir = Path(output_dir)\n    comparison = comparison_rows(evaluations)\n    diagnosis = diagnosis_rows(evaluations)\n    save_csv(output_dir / "comparacao_modelos.csv", comparison)\n    save_csv(output_dir / "diagnostico_erros.csv", diagnosis)\n    save_csv(output_dir / "bootstrap_pareado.csv", bootstrap)\n    gallery = []\n    for cohort, models in evaluations.items():\n        records = {r["id"]: r for r in cohorts[cohort]}\n        caches = {name: load_prediction_cache(result["cache_file"]) for name, result in models.items()}\n        cases = select_gallery_cases(models["antigo"]["images"], models["candidato"]["images"], seed=seed)\n        old_images = {r["id"]: r for r in models["antigo"]["images"]}\n        new_images = {r["id"]: r for r in models["candidato"]["images"]}\n        for i, case in enumerate(cases):\n            record = records[case["id"]]\n            relative = f"galeria/{cohort}_{i:03}.jpg"\n            render_case(record, caches["antigo"][record["id"]],\n                        caches["candidato"][record["id"]], policy, output_dir/relative)\n            gallery.append({**case, "coorte": cohort, "imagem": record["name"], "arquivo": relative,\n                            "antigo": {k: old_images[record["id"]][k] for k in\n                                       ["tp_alerta", "fn_alerta", "fp_alerta", "alta_sem_alerta"]},\n                            "candidato": {k: new_images[record["id"]][k] for k in\n                                          ["tp_alerta", "fn_alerta", "fp_alerta", "alta_sem_alerta"]},\n                            "creditos": record.get("creditos", {})})\n    save_json(output_dir / "indice_galeria.json", gallery)\n    new_records = cohorts["openimages_val_nova"]\n    save_csv(output_dir / "revisao_cenarios.csv", [\n        {"id": r["id"], "imagem": r["name"], "ambiente_interno": "nao_revisado",\n         "proximo_do_uso_pretendido": "nao_revisado", "anotacao_parece_correta": "nao_revisado",\n         "observacao": "", "origem": r.get("creditos", {}).get("OriginalLandingURL", "")}\n        for r in new_records])\n    provenance = audit["baseline_original"]["estado_proveniencia_original"]\n    table = "<table><thead><tr><th>Conjunto</th><th>Modelo</th><th>Precisão alertas</th><th>Recall alertas</th><th>F1</th><th>FN</th><th>Alta sem alerta</th></tr></thead><tbody>"\n    for r in comparison:\n        table += "<tr>" + "".join("<td>"+html.escape(str(x))+"</td>" for x in\n                     [r["coorte"], r["modelo"], format_percent(r["precision_alertas"]),\n                      format_percent(r["recall_alertas"]), format_percent(r["f1_alertas"]),\n                      r["fn_alerta"], r["alta_sem_alerta"]]) + "</tr>"\n    table += "</tbody></table>"\n    cards = []\n    for case in gallery:\n        credit = case["creditos"]\n        source_url = str(credit.get("OriginalLandingURL", ""))\n        license_url = str(credit.get("License", ""))\n        attribution = html.escape(str(credit.get("Author", "")))\n        if source_url.startswith(("https://", "http://")):\n            attribution += \' · <a href="\'+html.escape(source_url, quote=True)+\'">Fonte original</a>\'\n        if license_url.startswith(("https://", "http://")):\n            attribution += \' · <a href="\'+html.escape(license_url, quote=True)+\'">Licença</a>\'\n        details = (f"Antigo: FN {case[\'antigo\'][\'fn_alerta\']}, FP {case[\'antigo\'][\'fp_alerta\']}, "\n                   f"alta sem alerta {case[\'antigo\'][\'alta_sem_alerta\']}. "\n                   f"Candidato: FN {case[\'candidato\'][\'fn_alerta\']}, FP {case[\'candidato\'][\'fp_alerta\']}, "\n                   f"alta sem alerta {case[\'candidato\'][\'alta_sem_alerta\']}.")\n        cards.append("<article><h3>"+html.escape(case["coorte"]+" / "+case["categoria"])+\n                     "</h3><p>"+html.escape(case["imagem"]+" — "+details)+"</p>"+\n                     \'<img loading="lazy" src="\'+case["arquivo"]+\'" alt="Caixas de referência e dos dois modelos">\'+\n                     "<p class=small>"+attribution+\n                     " · Sobreposição de caixas para avaliação.</p></article>")\n    html_report = """<!doctype html><html lang="pt-BR"><meta charset="utf-8">\n<meta name="viewport" content="width=device-width,initial-scale=1">\n<title>TCC — Diagnóstico dos detectores</title><style>\nbody{font:16px/1.6 system-ui,sans-serif;background:#f3f5f7;color:#142135;margin:0}\nmain{max-width:1400px;margin:auto;padding:32px}h1{font-size:32px;line-height:1.2}\n.note,article,table{background:white;border:1px solid #dbe1e8;border-radius:12px}\n.note,article{padding:20px;margin:20px 0}table{border-collapse:collapse;width:100%}\ntd,th{padding:10px;text-align:left;border-bottom:1px solid #e6eaf0}th{background:#e8eff8}\nimg{max-width:100%;height:auto}.small{font-size:12px;color:#526174}.scroll{overflow:auto}\n</style><main><h1>Diagnóstico comparativo dos detectores</h1>\n<p>Pesos congelados · 12 classes · política V2 · confiança 0,25 · sem treinamento</p>\n<div class=note><strong>Como interpretar</strong><p>\nA Home VAL é histórica e exploratória. A validação externa é nova em relação aos IDs\ne bytes dos dados conhecidos do projeto; pré-treinamentos públicos não foram auditados.\nO recorte Open Images tem cobertura parcial e não garante cenas internas.\nAs métricas de alertas medem concordância com a V2, não risco físico independente.\nA reserva futura não foi inferida. Nenhum modelo foi promovido automaticamente.</p>\n<p>Proveniência documental do baseline: <strong>""" + html.escape(provenance) + """</strong>.</p></div>\n<h2>Alertas por objeto avaliável</h2><div class=scroll>""" + table + """</div>\n<p>Os CSVs trazem suporte por classe, AP no recorte, contagens e intervalos pareados.\nNão compare APs de protocolos e distribuições diferentes como se fossem o mesmo benchmark.</p>\n<h2>Casos para inspeção</h2><p>Os casos abaixo priorizam diferenças e erros, além de uma amostra por hash.\nEsta galeria não é uma estimativa da frequência dos erros. As contagens acima usam todas as imagens.\nObjetos sem cobertura aparecem em cinza e não são classificados automaticamente como falsos positivos.</p>\n""" + "".join(cards) + """<h2>Próxima decisão</h2><p>\nRevisar os cenários e os indícios dos erros; só depois definir o treino de controle.\nOs pesos antigos, o candidato e os testes anteriores permanecem preservados.</p></main></html>"""\n    (output_dir / "relatorio_diagnostico.html").write_text(html_report, encoding="utf-8")\n    lines = ["# Diagnóstico 11", "", "Comparação sem novo treinamento e sem seleção automática.",\n             "", f"Proveniência original do baseline: {provenance}.",\n             f"Validação externa: {len(new_records)} imagens.",\n             f"Reserva futura não inferida: {dataset_manifest[\'contagens\'].get(\'reserva_futura\', 0)} imagens.",\n             "", "| Conjunto | Modelo | Precisão alertas | Recall alertas | F1 | FN | Alta sem alerta |",\n             "|---|---|---:|---:|---:|---:|---:|"]\n    for r in comparison:\n        lines.append("| "+" | ".join(map(str, [r["coorte"], r["modelo"],\n                     format_percent(r["precision_alertas"]), format_percent(r["recall_alertas"]),\n                     format_percent(r["f1_alertas"]), r["fn_alerta"], r["alta_sem_alerta"]]))+" |")\n    lines += ["", "A nova cobertura é conservadora e a seleção busca diversidade de classes; não representa a frequência natural do uso.",\n              "A Home VAL continua exploratória enquanto a origem do primeiro treinamento não estiver esclarecida.",\n              "Semelhança por pHash é um filtro de suspeitas, não uma prova de identidade de cena.",\n              "Os indícios de erro podem se sobrepor e não identificam causas de forma conclusiva.",\n              "A reserva atual não foi inferida; não há avaliação independente de risco físico.",\n              "", "Veja relatorio_diagnostico.html para a galeria e bootstrap_pareado.csv para a incerteza das diferenças."]\n    (output_dir / "relatorio_diagnostico.md").write_text("\\n".join(lines)+"\\n", encoding="utf-8")\n    return comparison, diagnosis, gallery\n\n\ndef package_reports(output_dir):\n    output_dir = Path(output_dir)\n    archive = output_dir / "pacote_diagnostico_11.zip"\n    with zipfile.ZipFile(archive, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=4) as z:\n        for path in sorted(output_dir.rglob("*")):\n            if path.is_file() and path != archive and not path.name.endswith(".tmp"):\n                z.write(path, path.relative_to(output_dir))\n    return archive\n', 'exp12_protocol.py': '"""Orçamento e seleção do experimento 12, sem dependência de Torch."""\nfrom __future__ import annotations\n\nfrom collections import Counter\nimport hashlib\nimport math\nimport numpy as np\n\nARMS = {\n    "A_home": {"home_per_batch": 16, "oi_per_batch": 0},\n    "B_home_replay": {"home_per_batch": 12, "oi_per_batch": 4},\n}\n\n\nclass BudgetSampler:\n    """Fluxos por fonte independentes; quantidade fixa de batches por época."""\n    def __init__(self, home, oi, batch_size, batches, oi_per_batch, seed=42):\n        self.home, self.oi = list(home), list(oi)\n        self.batch_size, self.batches = int(batch_size), int(batches)\n        self.oi_per_batch = int(oi_per_batch)\n        self.home_per_batch = self.batch_size - self.oi_per_batch\n        self.seed, self.epoch = int(seed), 0\n        if (not self.home or self.batches <= 0 or self.batch_size <= 0 or\n                not 0 <= self.oi_per_batch < self.batch_size or\n                (self.oi_per_batch and not self.oi) or set(self.home) & set(self.oi)):\n            raise ValueError("Fontes ou orçamento inválidos.")\n\n    def __len__(self):\n        return self.batches*self.batch_size\n\n    def set_epoch(self, epoch):\n        self.epoch = int(epoch)\n\n    def _stream(self, values, needed, source):\n        if not needed:\n            return []\n        rng = np.random.default_rng(np.random.SeedSequence([self.seed, self.epoch, source]))\n        result = []\n        while len(result) < needed:\n            result.extend(rng.permutation(values).tolist())\n        return result[:needed]\n\n    def __iter__(self):\n        home = self._stream(self.home, self.batches*self.home_per_batch, 0)\n        oi = self._stream(self.oi, self.batches*self.oi_per_batch, 1)\n        rng = np.random.default_rng(np.random.SeedSequence([self.seed, self.epoch, 2]))\n        for b in range(self.batches):\n            batch = home[b*self.home_per_batch:(b+1)*self.home_per_batch]\n            batch += oi[b*self.oi_per_batch:(b+1)*self.oi_per_batch]\n            rng.shuffle(batch)\n            yield from batch\n\n\ndef order_id(iid, seed=42):\n    return hashlib.sha256(f"{seed}|{iid}".encode()).hexdigest(), iid\n\n\ndef diverse_subset(rows, n=2000, seed=42, per_class=40):\n    if n <= 0 or n > len(rows) or len({r["id"] for r in rows}) != len(rows):\n        raise ValueError("Tamanho do replay ou identidade dos IDs inválidos.")\n    mapping = {r["id"]: r for r in rows}\n    groups = {c: sorted([r["id"] for r in rows if int(r["classe_primaria"]) == c],\n                        key=lambda iid: order_id(iid, seed)) for c in range(12)}\n    chosen = set()\n    for k in range(per_class):\n        for c in sorted(groups, key=lambda cid: (len(groups[cid]), cid)):\n            if k < len(groups[c]) and len(chosen) < n:\n                chosen.add(groups[c][k])\n    for iid in sorted(mapping, key=lambda x: order_id(x, seed)):\n        if len(chosen) == n:\n            break\n        chosen.add(iid)\n    return [mapping[iid] for iid in sorted(chosen)]\n\n\ndef choose_replay(history_rows, n=2200, seed=42, per_class=40, forbidden_hashes=()):\n    batches = {f"lote_{i:06}.zip" for i in range(1, 6)}\n    rows = [r for r in history_rows if r["lote"] in batches]\n    if len(rows) != 5000 or any(r["split"] != "train" for r in rows):\n        raise ValueError("Os cinco lotes do piloto precisam conter 5.000 imagens só de treino.")\n    forbidden = set(forbidden_hashes)\n    return diverse_subset([r for r in rows if r["imagem_sha256"] not in forbidden],\n                          n, seed, per_class)\n\n\ndef validate_metric_block(block, label):\n    for k in ["precision", "recall", "f1"]:\n        x = block["alertas_por_objeto"][k]\n        if x is None or not math.isfinite(x):\n            raise ValueError(f"Métrica sem suporte ou não finita: {label}/{k}")\n    for k in ["alta_ref", "alta_sem_alerta"]:\n        if int(block[k]) < 0:\n            raise ValueError("Contagem inválida.")\n\n\ndef make_rule(baseline_home, candidate_external, dataset_sha, policy_sha):\n    validate_metric_block(baseline_home, "baseline_home")\n    validate_metric_block(candidate_external, "candidate_external")\n    return {\n        "versao": "12-regra-duas-validacoes-v1",\n        "home": {\n            "precision_maior_que": baseline_home["alertas_por_objeto"]["precision"],\n            "recall_minimo": baseline_home["alertas_por_objeto"]["recall"],\n            "alta_sem_alerta_maximo": baseline_home["alta_sem_alerta"],\n            "objetos_alerta_ref": baseline_home["alertas_por_objeto"]["tp"]+baseline_home["alertas_por_objeto"]["fn"],\n            "alta_ref": baseline_home["alta_ref"],\n        },\n        "externa": {\n            "precision_minima": candidate_external["alertas_por_objeto"]["precision"],\n            "recall_minimo": candidate_external["alertas_por_objeto"]["recall"],\n            "alta_sem_alerta_maximo": candidate_external["alta_sem_alerta"],\n            "objetos_alerta_ref": candidate_external["alertas_por_objeto"]["tp"]+candidate_external["alertas_por_objeto"]["fn"],\n            "alta_ref": candidate_external["alta_ref"],\n        },\n        "dataset_sha256": dataset_sha, "politica_sha256": policy_sha,\n        "conf_alertas": .25, "testes_consultados": False,\n        "tolerancia_numerica": 1e-12,\n        "criterio": "Home supera precisão do baseline, preserva recall e omissões altas; externa preserva precisão, recall e omissões altas do candidato 10.",\n        "desempate": "Precisão Home, recall Home, omissões altas Home, F1 externa, menos atualizações, identificador.",\n        "sem_elegivel": "Preservar baseline em uso; candidato 10 e todos os checkpoints continuam disponíveis.",\n    }\n\n\ndef judge(home, external, rule):\n    validate_metric_block(home, "home")\n    validate_metric_block(external, "external")\n    h, e = home["alertas_por_objeto"], external["alertas_por_objeto"]\n    rh, re = rule["home"], rule["externa"]\n    for observed, expected in [(h["tp"]+h["fn"], rh["objetos_alerta_ref"]),\n                               (e["tp"]+e["fn"], re["objetos_alerta_ref"]),\n                               (home["alta_ref"], rh["alta_ref"]),\n                               (external["alta_ref"], re["alta_ref"])]:\n        if observed != expected:\n            raise ValueError("Os denominadores de referência mudaram entre checkpoints.")\n    eps = rule["tolerancia_numerica"]\n    checks = {\n        "home_precision_superior": h["precision"] > rh["precision_maior_que"]+eps,\n        "home_recall_preservado": h["recall"] >= rh["recall_minimo"]-eps,\n        "home_altas_preservadas": home["alta_sem_alerta"] <= rh["alta_sem_alerta_maximo"],\n        "externa_precision_preservada": e["precision"] >= re["precision_minima"]-eps,\n        "externa_recall_preservado": e["recall"] >= re["recall_minimo"]-eps,\n        "externa_altas_preservadas": external["alta_sem_alerta"] <= re["alta_sem_alerta_maximo"],\n    }\n    return {**checks, "elegivel": all(checks.values()),\n            "retencao_externa": all(v for k, v in checks.items() if k.startswith("externa_"))}\n\n\ndef select_checkpoints(entries, rule):\n    if not entries:\n        raise ValueError("Nenhum checkpoint avaliado.")\n    assessed = []\n    for entry in entries:\n        checks = judge(entry["home"], entry["external"], rule)\n        assessed.append({**entry, "checks": checks})\n    eligible = [r for r in assessed if r["checks"]["elegivel"]]\n\n    def ranking(r):\n        h = r["home"]["alertas_por_objeto"]\n        return (-h["precision"], -h["recall"], r["home"]["alta_sem_alerta"],\n                -r["external"]["alertas_por_objeto"]["f1"], r["updates"], r["id"])\n\n    selected = min(eligible, key=ranking) if eligible else None\n    # Apenas para inspeção. A classificação exploratória não muda os requisitos.\n    retained = [r for r in assessed if r["checks"]["retencao_externa"]]\n    exploratory = min(retained, key=lambda r: (\n        -r["home"]["alertas_por_objeto"]["recall"],\n        r["home"]["alta_sem_alerta"],\n        -r["home"]["alertas_por_objeto"]["precision"],\n        -r["external"]["alertas_por_objeto"]["f1"],\n        r["updates"], r["id"])) if retained else None\n    return {"eligible": selected, "exploratory": exploratory, "assessed": assessed}\n', 'exp12_data.py': '"""Entradas congeladas e replay somente do TRAIN histórico."""\nfrom __future__ import annotations\n\nfrom collections import Counter\nimport csv\nimport gzip\nimport hashlib\nimport io\nimport json\nfrom pathlib import Path\nimport zipfile\n\nimport numpy as np\nfrom PIL import Image\nimport yaml\n\nimport diag11_core as core\nimport diag11_data as d11\nfrom exp12_protocol import diverse_subset\n\n\ndef check_digest(raw, expected, label):\n    if hashlib.sha256(raw).hexdigest() != expected:\n        raise RuntimeError("Conteúdo alterado: " + label)\n\n\ndef materialise_home_labels(zip_path, home):\n    with zipfile.ZipFile(zip_path) as z:\n        for record in home:\n            member = record["id"].removeprefix("home:")\n            label_name = member.replace("/images/", "/labels/").rsplit(".", 1)[0]+".txt"\n            core.safe_member(label_name)\n            raw = z.read(label_name)\n            check_digest(raw, record["label_sha256"], label_name)\n            label = Path(str(record["image"]).replace("/images/", "/labels/")).with_suffix(".txt")\n            label.parent.mkdir(parents=True, exist_ok=True)\n            label.write_bytes(raw)\n            record["label"] = label\n\n\ndef read_package11(path, expected_sha, expected_models, expected_dataset):\n    if core.sha256_file(path) != expected_sha:\n        raise RuntimeError("O pacote do diagnóstico 11 mudou.")\n    with zipfile.ZipFile(path) as z:\n        if len(z.namelist()) != len(set(z.namelist())):\n            raise RuntimeError("Pacote 11 com membros repetidos.")\n        for name in z.namelist():\n            core.safe_member(name)\n        summary = json.loads(z.read("resumo_diagnostico_11.json"))\n        config = json.loads(z.read("configuracao_diagnostico.json"))\n        dataset_location = json.loads(z.read("localizacao_dataset.json"))\n        manifest = json.loads(z.read("manifesto_validacao_nova.json"))\n        if (config["modelos_sha256"] != expected_models or\n                config["dataset_sha256"] != expected_dataset or\n                dataset_location["dataset_sha256"] != expected_dataset or\n                manifest["dataset_sha256"] != expected_dataset or\n                summary["reserva_futura_inferida"] is not False):\n            raise RuntimeError("Identidades ou reserva do diagnóstico 11 incompatíveis.")\n    return summary, config, dataset_location, manifest\n\n\ndef load_frozen_validation(zip_path, destination, expected_archive_sha, expected_dataset_sha,\n                           expected_manifest):\n    if core.sha256_file(zip_path) != expected_archive_sha:\n        raise RuntimeError("ZIP da validação independente diferente do congelado no 11.")\n    destination = Path(destination)\n    val, reserved_ids, frozen_files = [], set(), []\n    with zipfile.ZipFile(zip_path) as z:\n        if len(z.namelist()) != len(set(z.namelist())):\n            raise RuntimeError("Dataset novo com membros repetidos.")\n        for member in z.namelist():\n            core.safe_member(member)\n        manifest = json.loads(z.read("manifesto_validacao_nova.json"))\n        if manifest != expected_manifest:\n            raise RuntimeError("O manifesto do pacote difere do diagnóstico 11.")\n        raw_manifest = dict(manifest)\n        claimed = raw_manifest.pop("dataset_sha256")\n        if hashlib.sha256(core.canonical(raw_manifest).encode()).hexdigest() != claimed or claimed != expected_dataset_sha:\n            raise RuntimeError("Hash canônico da validação nova não confere.")\n        if manifest["nomes"] != core.NAMES or manifest["origem_split"] != "validation":\n            raise RuntimeError("Classes ou origem da validação alteradas.")\n        items = manifest["arquivos"]\n        if (Counter(r["role"] for r in items) != {"val_nova": 1000, "reserva_futura": 300}\n                or len({r["id"] for r in items}) != 1300\n                or len({r["sha256"] for r in items}) != 1300):\n            raise RuntimeError("Identidades ou quantidades do conjunto independente incompatíveis.")\n        for item in items:\n            role = item["role"]\n            for field, prefix in [("image", "images"), ("label", "labels"), ("coverage", "coverage")]:\n                if not item[field].startswith(prefix+"/"+role+"/"):\n                    raise RuntimeError("Caminho do dataset não corresponde ao papel declarado.")\n                if item[field] not in z.namelist():\n                    raise RuntimeError("Arquivo declarado ausente no ZIP.")\n            if role == "reserva_futura":\n                reserved_ids.add(item["id"])\n                continue  # Nenhum JPEG, label ou cobertura da reserva é extraído.\n            paths = {}\n            for field, hash_field in [("image", "sha256"), ("label", "label_sha256"),\n                                      ("coverage", "coverage_sha256")]:\n                raw = z.read(item[field])\n                check_digest(raw, item[hash_field], item[field])\n                output = destination/item[field]\n                output.parent.mkdir(parents=True, exist_ok=True)\n                output.write_bytes(raw)\n                paths[field] = output\n                frozen_files.append((output, item[hash_field]))\n            coverage = json.loads(paths["coverage"].read_text())\n            if any(coverage[k] != item[k] for k in ["known_mask", "positive_mask", "negative_mask"]):\n                raise RuntimeError("Cobertura da validação foi alterada.")\n            with Image.open(paths["image"]) as image:\n                if image.size != (item["width"], item["height"]):\n                    raise RuntimeError("Dimensões diferentes do manifesto.")\n                image.verify()\n            parsed = core.parse_yolo(paths["label"].read_bytes(), item["width"], item["height"])\n            if len(parsed) != len(item["gt_normalizado"]):\n                raise RuntimeError("Quantidade de caixas difere do manifesto.")\n            gt = []\n            for original, label_obj in zip(item["gt_normalizado"], parsed):\n                a, b, c, d = original["bbox_norm"]\n                box = [a*item["width"], b*item["height"], c*item["width"], d*item["height"]]\n                if (original["classe_id"] != label_obj["classe_id"] or\n                        not np.allclose(box, label_obj["bbox"], rtol=0, atol=.001)):\n                    raise RuntimeError("Caixa do manifesto diverge do label autenticado.")\n                gt.append({**original, "bbox": box})\n            val.append({**item, **paths, "gt": gt, "role": "val_nova",\n                        "coorte": "openimages_val_nova", "name": paths["image"].name})\n    return val, reserved_ids, frozen_files\n\n\ndef load_replay(package_items, selected_history, local_dir, fetch_package, expected_cfg,\n                forbidden_ids, forbidden_hashes, protected_near, validator, final_size=2000, seed=42):\n    """Lê só os membros escolhidos dos cinco lotes existentes, com SHA integral."""\n    local_dir = Path(local_dir)\n    local_dir.mkdir(parents=True, exist_ok=True)\n    wanted = {r["id"]: r for r in selected_history}\n    protected_ids, protected_hashes = set(forbidden_ids), set(forbidden_hashes)\n    records, exclusions = {}, []\n    frozen, seen, package_receipts = [], set(), []\n    for item in package_items:\n        pkg = local_dir/item["arquivo"]\n        fetch_package(item, pkg)\n        if pkg.stat().st_size != item["zip_bytes"] or core.sha256_file(pkg) != item["zip_sha256"]:\n            raise RuntimeError("Pacote de replay incompleto ou alterado.")\n        with zipfile.ZipFile(pkg) as z:\n            if len(z.namelist()) != len(set(z.namelist())):\n                raise RuntimeError("Lote de replay com membros repetidos.")\n            for name in z.namelist():\n                core.safe_member(name)\n            manifest = json.loads(z.read("manifesto_lote.json"))\n            if manifest["config_sha256"] != expected_cfg or manifest["origem_split"] != "train":\n                raise RuntimeError("Lote de replay não pertence ao TRAIN autenticado.")\n            for r in manifest["aceitas"]:\n                if r["id"] not in wanted:\n                    continue\n                if r["id"] in seen:\n                    raise RuntimeError("ID de replay repetido entre lotes.")\n                seen.add(r["id"])\n                prior = wanted[r["id"]]\n                if (prior["split"] != "train" or prior["lote"] != item["arquivo"] or\n                        prior["imagem_sha256"] != r["sha256"] or\n                        int(prior["known_mask"]) != r["known_mask"]):\n                    raise RuntimeError("Replay diverge da divisão 80/20 histórica.")\n                if r["id"] in protected_ids or r["sha256"] in protected_hashes:\n                    raise RuntimeError("Replay coincide com dado protegido.")\n                raw_image, raw_label, raw_cov = (z.read(r[k]) for k in ["imagem", "label", "cobertura"])\n                for raw, key in [(raw_image, "sha256"), (raw_label, "label_sha256"), (raw_cov, "cobertura_sha256")]:\n                    check_digest(raw, r[key], r["id"]+"/"+key)\n                classes = validator(raw_label, source=item["arquivo"]+":"+r["label"])\n                cov = json.loads(raw_cov)\n                known_classes = (set(cov["classes_positivas_verificadas_exaustivas"]) |\n                                 set(cov["classes_negativas_verificadas"]))\n                if (not all(isinstance(cid, int) and 0 <= cid < 12 for cid in known_classes)\n                        or sum(1 << cid for cid in known_classes) != r["known_mask"]\n                        or cov["known_mask"] != r["known_mask"]\n                        or set(classes) & set(cov["classes_negativas_verificadas"])):\n                    raise RuntimeError("Máscara ou anotação parcial de replay incompatível.")\n                image = local_dir/"images/train"/f"oi_{r[\'id\']}.jpg"\n                label = local_dir/"labels/train"/f"oi_{r[\'id\']}.txt"\n                coverage_file = local_dir/"coverage/train"/f"oi_{r[\'id\']}.json"\n                for path, raw in [(image, raw_image), (label, raw_label), (coverage_file, raw_cov)]:\n                    path.parent.mkdir(parents=True, exist_ok=True)\n                    path.write_bytes(raw)\n                with Image.open(image) as im:\n                    width, height = im.size\n                    im.verify()\n                if (width, height) != (r["width"], r["height"]):\n                    raise RuntimeError("Dimensões de replay incompatíveis.")\n                near = protected_near.find(d11.phash(image))\n                if near:\n                    exclusions.append({"id": r["id"], "motivo": "semelhanca_phash_com_validacao_ou_reserva", "semelhante_a": near})\n                    continue\n                gt = core.parse_yolo(raw_label, width, height)\n                records[r["id"]] = {\n                    **prior, "oi_id": r["id"], "id": r["id"], "source": "openimages",\n                    "image": image, "label": label, "coverage": coverage_file,\n                    "known_mask": r["known_mask"], "sha256": r["sha256"],\n                    "label_sha256": r["label_sha256"], "coverage_sha256": r["cobertura_sha256"],\n                    "width": width, "height": height, "classes_com_caixa": classes,\n                    "gt": gt, "name": image.name, "role": "train", "coorte": "replay_train",\n                }\n        package_receipts.append(item)\n        pkg.unlink()\n    if seen != set(wanted):\n        raise RuntimeError("Alguns IDs planejados não foram encontrados nos cinco lotes.")\n    if len(records) < final_size:\n        raise RuntimeError(f"Replay com somente {len(records)} elegíveis; exigidos {final_size}.")\n    chosen = diverse_subset(list(records.values()), final_size, seed)\n    output = {Path(r["image"]).resolve().as_posix(): r for r in chosen}\n    for record in output.values():\n        frozen += [(record["image"], record["sha256"]), (record["label"], record["label_sha256"]),\n                   (record["coverage"], record["coverage_sha256"])]\n    return output, exclusions, frozen, package_receipts\n\n\ndef make_training_inputs(home, replay, destination):\n    destination = Path(destination)\n    destination.mkdir(parents=True, exist_ok=True)\n    home_train = {Path(r["image"]).resolve().as_posix():\n                  {**r, "source": "home", "known_mask": core.ALL_KNOWN}\n                  for r in home if r["role"] == "home_train"}\n    val = [Path(r["image"]).resolve().as_posix() for r in home if r["role"] == "home_val"]\n    if len(home_train) != 1824 or len(val) != 456:\n        raise RuntimeError("Home diferente da configuração congelada.")\n    val_file = destination/"home_val.txt"\n    val_file.write_text("\\n".join(sorted(val))+"\\n", encoding="utf-8")\n    arms = {}\n    for name, records in [("A_home", home_train), ("B_home_replay", {**home_train, **replay})]:\n        path = destination/(name+".txt")\n        path.write_text("\\n".join(sorted(records))+"\\n", encoding="utf-8")\n        data_file = destination/(name+".yaml")\n        data_file.write_text(yaml.safe_dump({"path": str(destination.resolve()), "train": str(path.resolve()),\n                                           "val": str(val_file.resolve()), "names": dict(enumerate(core.NAMES))},\n                                          sort_keys=False, allow_unicode=True), encoding="utf-8")\n        arms[name] = {"records": records, "yaml": data_file}\n    return arms\n\n\ndef restore_result(package_path, cohort, model, cache_dir):\n    cache_dir = Path(cache_dir)/cohort/model\n    cache_dir.mkdir(parents=True, exist_ok=True)\n    with zipfile.ZipFile(package_path) as z:\n        prefix = cohort+"/"+model+"/"\n        summary = json.loads(z.read(prefix+"metricas.json"))\n        def rows(name):\n            return list(csv.DictReader(io.StringIO(z.read(prefix+name).decode("utf-8-sig"))))\n        images, events, classes = rows("alertas_por_imagem.csv"), rows("avaliacao_objetos.csv"), rows("metricas_por_classe.csv")\n        for row in images:\n            for k in ["tp_alerta", "fn_alerta", "fp_alerta", "alta_ref", "alta_sem_alerta", "tp_detector",\n                      "fn_detector", "fp_detector", "gt_avaliaveis", "gt_ignorados", "predicoes_ignoradas_conf_025"]:\n                row[k] = int(row[k])\n            for k in ["cobertura_completa_alertas", "ha_alerta_ref", "ha_alerta_pred"]:\n                row[k] = row[k] == "True"\n        for row in events:\n            row["classe_id"] = int(row["classe_id"])\n            for k in ["ha_alerta_ref", "ha_alerta_pred"]:\n                row[k] = row[k] == "True"\n        cache = cache_dir/"predicoes_conf_0001.jsonl.gz"\n        cache.write_bytes(z.read(prefix+"predicoes_conf_0001.jsonl.gz"))\n    return {"summary": summary, "images": images, "events": events, "classes": classes, "cache_file": cache}\n\n\ndef verify_frozen(files):\n    for path, sha in files:\n        if core.sha256_file(path) != sha:\n            raise RuntimeError("Arquivo protegido alterado: "+str(path))\n', 'conf14_data.py': '"""Entradas autenticadas do diagnóstico 14; extrai somente Home VAL."""\nfrom __future__ import annotations\nfrom collections import Counter\nimport csv\nimport hashlib\nimport io\nimport json\nfrom pathlib import Path\nimport zipfile\nfrom PIL import Image\nimport diag11_core as core\n\nCOHORTS = (\'home_val_historica\', \'openimages_val_nova\')\nBOOLS = {\'True\':True, \'False\':False}\n\ndef rows(raw):\n    return list(csv.DictReader(io.StringIO(raw.decode(\'utf-8-sig\'))))\n\ndef parse_events(raw):\n    result = rows(raw)\n    for r in result:\n        r[\'classe_id\'] = int(r[\'classe_id\'])\n        for field in [\'bbox_ref\',\'bbox_pred\']:\n            r[field] = json.loads(r[field]) if r[field] else None\n        for field in [\'ha_alerta_ref\',\'ha_alerta_pred\']:\n            r[field] = BOOLS[r[field]]\n        r[\'confianca\'] = float(r[\'confianca\']) if r[\'confianca\'] else None\n    return result\n\ndef gt_key(r):\n    return (r[\'id\'],int(r[\'classe_id\']),tuple(round(float(v),6) for v in r[\'bbox_ref\']))\n\ndef gt_map(events):\n    result = {}\n    for r in events:\n        if r[\'bbox_ref\'] is None: continue\n        key = gt_key(r)\n        if key in result: raise ValueError(\'Objeto GT repetido nos registros.\')\n        result[key] = r\n    return result\n\ndef load_source12(path, expected, output=None):\n    if core.sha256_file(path)!=expected[\'pacote_12\']:\n        raise RuntimeError(\'O pacote 12 mudou em relação ao recebido.\')\n    bundle = {\'evaluations\':{}}\n    with zipfile.ZipFile(path) as z:\n        if len(z.namelist())!=len(set(z.namelist())):\n            raise RuntimeError(\'Pacote 12 com membros repetidos.\')\n        for name in z.namelist(): core.safe_member(name)\n        if z.testzip() is not None: raise RuntimeError(\'Falha de CRC no pacote 12.\')\n        cfg = json.loads(z.read(\'configuracao_experimento_12.json\'))\n        summary = json.loads(z.read(\'resumo_experimento_12.json\'))\n        if not summary[\'concluido\'] or summary[\'reserva_futura_inferida\'] or summary[\'test_home_inferido\'] or summary[\'test_oi20k_inferido\']:\n            raise RuntimeError(\'Execução 12 incompatível com o protocolo.\')\n        if cfg[\'expected\'][\'candidato\']!=expected[\'candidato\'] or cfg[\'expected\'][\'politica\']!=expected[\'politica\'] or summary[\'dataset_sha256\']!=expected[\'dataset_novo\']:\n            raise RuntimeError(\'Referências ou política diferentes do experimento 12.\')\n        if cfg[\'codigo_embutido_sha256\']!=expected[\'codigo_12\']:\n            raise RuntimeError(\'Código do experimento 12 diferente do recebido.\')\n        for name, digest in expected[\'codigo_12\'].items():\n            if hashlib.sha256(z.read(\'codigo/\'+name)).hexdigest()!=digest:\n                raise RuntimeError(\'Código alterado no pacote 12: \'+name)\n        rule_raw = z.read(\'regra_selecao_congelada.json\')\n        if hashlib.sha256(rule_raw).hexdigest()!=cfg[\'regra_sha256\']:\n            raise RuntimeError(\'A regra original do experimento 12 foi alterada.\')\n        selection = json.loads(z.read(\'selecao_checkpoints.json\'))\n        checkpoint = next(e for e in selection[\'assessed\'] if e[\'id\']==\'B_home_replay_e03\')\n        if checkpoint[\'weight_sha256\']!=expected[\'B_e03\']:\n            raise RuntimeError(\'Checkpoint B_e03 diferente do esperado.\')\n        for model,prefix in [(\'baseline\',\'referencias/baseline\'),(\'candidato_10\',\'referencias/candidato_10\'),(\'B_e03\',\'avaliacoes/B_home_replay_e03\')]:\n            bundle[\'evaluations\'][model] = {}\n            for cohort in COHORTS:\n                base=prefix+\'/\'+cohort+\'/\'\n                members={name:z.read(base+name) for name in [\'metricas.json\',\'alertas_por_imagem.csv\',\'avaliacao_objetos.csv\']}\n                metrics=json.loads(members[\'metricas.json\'])\n                images=rows(members[\'alertas_por_imagem.csv\'])\n                for r in images:\n                    for key in [\'tp_alerta\',\'fn_alerta\',\'fp_alerta\',\'alta_ref\',\'alta_sem_alerta\',\'tp_detector\',\'fn_detector\',\'fp_detector\',\'gt_avaliaveis\',\'gt_ignorados\',\'predicoes_ignoradas_conf_025\']:\n                        r[key]=int(r[key])\n                if len({r[\'id\'] for r in images})!=len(images) or len(images)!=metrics[\'imagens\']:\n                    raise RuntimeError(\'IDs ou quantidades inconsistentes nos registros 12.\')\n                for field, n in [(\'tp_alerta\',metrics[\'alertas_por_objeto\'][\'tp\']),(\'fn_alerta\',metrics[\'alertas_por_objeto\'][\'fn\']),(\'fp_alerta\',metrics[\'alertas_por_objeto\'][\'fp\']),(\'alta_ref\',metrics[\'alta_ref\']),(\'alta_sem_alerta\',metrics[\'alta_sem_alerta\'])]:\n                    if sum(r[field] for r in images)!=n: raise RuntimeError(\'Contagens inconsistentes no pacote 12.\')\n                events=parse_events(members[\'avaliacao_objetos.csv\'])\n                gt_map(events)\n                bundle[\'evaluations\'][model][cohort]={\'summary\':metrics,\'images\':images,\'events\':events}\n                if output is not None:\n                    dest=Path(output)/\'referencias_12\'/model/cohort\n                    dest.mkdir(parents=True,exist_ok=True)\n                    for name, raw in members.items(): (dest/name).write_bytes(raw)\n        for cohort in COHORTS:\n            target=gt_map(bundle[\'evaluations\'][\'B_e03\'][cohort][\'events\'])\n            for model in [\'baseline\',\'candidato_10\']:\n                source=gt_map(bundle[\'evaluations\'][model][cohort][\'events\'])\n                if source.keys()!=target.keys(): raise RuntimeError(\'Modelos do 12 usam referências GT diferentes.\')\n                if any(source[k][\'ha_alerta_ref\']!=target[k][\'ha_alerta_ref\'] or source[k][\'prioridade_ref\']!=target[k][\'prioridade_ref\'] for k in target):\n                    raise RuntimeError(\'Referências de prioridade diferentes entre modelos.\')\n        bundle.update(config=cfg,summary=summary,rule=json.loads(rule_raw),\n                      audit=json.loads(z.read(\'auditoria_dados.json\')))\n    return bundle\n\ndef extract_home_val(path, destination, expected_zip, expected_test, expected_policy):\n    if core.sha256_file(path)!=expected_zip: raise RuntimeError(\'ZIP Home alterado.\')\n    destination=Path(destination)\n    records=[];frozen=[]\n    with zipfile.ZipFile(path) as z:\n        if len(z.namelist())!=len(set(z.namelist())): raise RuntimeError(\'Membros repetidos no Home.\')\n        for name in z.namelist():core.safe_member(name)\n        manifest=json.loads(z.read(\'verificacao_preparacao.json\'))\n        if (manifest[\'test\'][\'sha256\']!=expected_test or manifest[\'test\'][\'imagens\']!=403 or\n            {int(k):v for k,v in manifest[\'classes\'].items()}!=dict(enumerate(core.NAMES))):\n            raise RuntimeError(\'Manifesto Home incompatível.\')\n        root=manifest[\'dataset_dir\']\n        items={r[\'arquivo\']:r for r in manifest[\'arquivos\']}\n        if len(items)!=len(manifest[\'arquivos\']): raise RuntimeError(\'Manifesto Home repetido.\')\n        train_hash=set();val_hash=set();all_hash=set();counts=Counter()\n        for name,item in items.items():\n            is_image=name.startswith(root+\'/images/\') and Path(name).suffix.lower() in {\'.jpg\',\'.jpeg\',\'.png\',\'.webp\',\'.bmp\'}\n            if not is_image:continue\n            all_hash.add(item[\'sha256\'])\n            if name.startswith(root+\'/images/train/raw_train/\'):\n                counts[\'train\']+=1;train_hash.add(item[\'sha256\'])\n            elif name.startswith(root+\'/images/val/raw_train/\'):\n                counts[\'val\']+=1;val_hash.add(item[\'sha256\'])\n        if counts!={\'train\':1824,\'val\':456} or train_hash&val_hash:\n            raise RuntimeError(\'A divisão atual de Home mudou ou tem bytes repetidos entre TRAIN e VAL.\')\n        for name in z.namelist():\n            if not(name.startswith(\'codigo/\') and name.endswith(\'.py\')):continue\n            raw=z.read(name)\n            if name in items and hashlib.sha256(raw).hexdigest()!=items[name][\'sha256\']:\n                raise RuntimeError(\'Código Home alterado.\')\n            out=destination/name;out.parent.mkdir(parents=True,exist_ok=True);out.write_bytes(raw)\n            frozen.append((out,hashlib.sha256(raw).hexdigest()))\n        for name,item in sorted(items.items()):\n            if not(name.startswith(root+\'/images/val/raw_train/\') and Path(name).suffix.lower() in {\'.jpg\',\'.jpeg\',\'.png\',\'.webp\',\'.bmp\'}):continue\n            label=name.replace(\'/images/\',\'/labels/\').rsplit(\'.\',1)[0]+\'.txt\'\n            if label not in items:raise RuntimeError(\'Label Home ausente.\')\n            paths={}\n            for field,member in [(\'image\',name),(\'label\',label)]:\n                raw=z.read(member)\n                if hashlib.sha256(raw).hexdigest()!=items[member][\'sha256\']:raise RuntimeError(\'Home alterado: \'+member)\n                out=destination/member;out.parent.mkdir(parents=True,exist_ok=True);out.write_bytes(raw)\n                paths[field]=out;frozen.append((out,items[member][\'sha256\']))\n            with Image.open(paths[\'image\']) as im:\n                width,height=im.size;im.verify()\n            records.append({\'id\':\'home:\'+name,\'name\':name.split(\'/raw_train/\')[-1],**paths,\n                \'role\':\'home_val\',\'coorte\':\'home_val_historica\',\'width\':width,\'height\':height,\n                \'known_mask\':core.ALL_KNOWN,\'sha256\':item[\'sha256\'],\'label_sha256\':items[label][\'sha256\'],\n                \'gt\':core.parse_yolo(paths[\'label\'].read_bytes(),width,height,source=label)})\n    if len(records)!=456 or core.sha256_file(destination/\'codigo/src/politica_alertas.py\')!=expected_policy:\n        raise RuntimeError(\'Quantidade Home VAL ou política V2 incompatível.\')\n    audit={\'home_train_manifesto\':1824,\'home_val_extraida\':456,\'home_train_extraida\':False,\n           \'home_test_extraido\':False,\'test_home_manifesto\':403,\'intersecao_bytes_train_val\':0,\n           \'historico_original_baseline_nao_comprovado_pela_separacao_atual\':True}\n    return records,all_hash,frozen,audit\n\ndef validate_cohorts(cohorts,bundle,reserved_ids):\n    if set(cohorts)!=set(COHORTS) or {k:len(v) for k,v in cohorts.items()}!={\'home_val_historica\':456,\'openimages_val_nova\':1000}:\n        raise ValueError(\'Coortes diferentes das duas validações congeladas.\')\n    for cohort,records in cohorts.items():\n        ids={r[\'id\'] for r in records}\n        if len(ids)!=len(records) or ids&set(reserved_ids):raise ValueError(\'ID repetido ou reservado na avaliação.\')\n        expected_role=\'home_val\' if cohort==\'home_val_historica\' else \'val_nova\'\n        if any(r[\'role\']!=expected_role for r in records):raise ValueError(\'Papel incompatível com a coorte de validação.\')\n        old=bundle[\'evaluations\'][\'B_e03\'][cohort]\n        if ids!={r[\'id\'] for r in old[\'images\']}:raise ValueError(\'IDs diferentes dos avaliados no 12.\')\n        actual={}\n        for r in records:\n            for g in r[\'gt\']:\n                if r[\'known_mask\']&(1<<g[\'classe_id\']):\n                    k=(r[\'id\'],g[\'classe_id\'],tuple(round(float(x),6) for x in g[\'bbox\']))\n                    if k in actual:raise ValueError(\'GT repetido na entrada atual.\')\n                    actual[k]=True\n        if actual.keys()!=gt_map(old[\'events\']).keys():raise ValueError(\'Caixas avaliáveis diferentes das referências originais.\')\n    return True\n', 'conf14_evaluation.py': '"""Varredura de confiança: uma inferência por imagem, política V2 fixa."""\nfrom __future__ import annotations\nfrom collections import Counter\nimport gzip\nimport hashlib\nimport json\nimport math\nfrom pathlib import Path\nimport diag11_core as core\nimport conf14_data as data\nfrom exp12_protocol import judge\n\nTHRESHOLDS=(.05,.10,.15,.20,.25,.30,.40)\nPREDICT_ARGS={\'conf\':.001,\'imgsz\':640,\'rect\':True,\'iou\':.70,\'max_det\':300,\'half\':False,\'batch\':1}\nVERSION=\'14-confianca-automatica-1.0\'\n\ndef point_name(conf):return \'conf_\'+f\'{conf:.2f}\'.replace(\'.\',\'\')\n\n\ndef cache_request(records,weight_sha,environment):\n    if not records or len({r[\'id\'] for r in records})!=len(records):raise ValueError(\'Cache exige IDs únicos.\')\n    if any(r[\'role\'] not in {\'home_val\',\'val_nova\'} for r in records):raise ValueError(\'Cache somente para validação.\')\n    signature=[{k:r[k] for k in [\'id\',\'sha256\',\'label_sha256\',\'width\',\'height\',\'known_mask\',\'role\']} for r in records]\n    return {\'versao\':VERSION,\'peso_sha256\':weight_sha,\'parametros\':PREDICT_ARGS,\n            \'ambiente\':environment,\'coorte\':records[0][\'coorte\'],\'imagens\':len(records),\n            \'entradas_sha256\':hashlib.sha256(core.canonical(signature).encode()).hexdigest()}\n\ndef read_cache(path,records):\n    expected={r[\'id\']:r for r in records};result={}\n    with gzip.open(path,\'rt\',encoding=\'utf-8\') as f:\n        for line in f:\n            row=json.loads(line);iid=row[\'id\']\n            if iid not in expected or iid in result:raise ValueError(\'Cache com ID desconhecido ou repetido.\')\n            r=expected[iid]\n            if row[\'image_sha256\']!=r[\'sha256\'] or row[\'label_sha256\']!=r[\'label_sha256\']:\n                raise ValueError(\'Cache pertence a outros bytes de imagem/label.\')\n            validate_predictions(row[\'predicoes\'],r);result[iid]=row[\'predicoes\']\n    if result.keys()!=expected.keys():raise ValueError(\'Cache incompleto.\')\n    return result\n\n\ndef event(record,kind,g,p,overlap,pred_low,conf):\n    row=core.record_event(record,\'B_e03\',kind,g,p,overlap,pred_low)\n    row[\'limiar_confianca\']=conf\n    row[\'melhor_confianca_mesma_classe\']=None\n    if kind==\'objeto_nao_detectado\':\n        flags=[v for v in row[\'indicios\'].split(\'|\') if v and v not in {\'deteccao_abaixo_da_confianca_025\',\'sobreposicao_com_previsao_de_outra_classe\',\'omissao_sem_indicio_automatico\'}]\n        matrix=core.iou_matrix([g],pred_low)[0]\n        same=[i for i,p0 in enumerate(pred_low) if p0[\'classe_id\']==g[\'classe_id\']]\n        if same:\n            best=max(same,key=lambda i:matrix[i]);row[\'melhor_confianca_mesma_classe\']=pred_low[best][\'confianca\']\n            if matrix[best]>=.5 and pred_low[best][\'confianca\']<conf:flags.append(\'deteccao_abaixo_do_limiar_avaliado\')\n        if any(matrix[i]>=.5 and p0[\'classe_id\']!=g[\'classe_id\'] and p0[\'confianca\']>=conf for i,p0 in enumerate(pred_low)):\n            flags.append(\'sobreposicao_com_previsao_de_outra_classe\')\n        row[\'indicios\']=\'|\'.join(flags or [\'omissao_sem_indicio_automatico\'])\n    return row\n\ndef analyse(record,pred_low,conf,policy,matcher):\n    if record[\'role\'] not in {\'home_val\',\'val_nova\'} or conf not in THRESHOLDS:\n        raise ValueError(\'Papel ou limiar fora do protocolo.\')\n    known=int(record[\'known_mask\'])\n    gt=core.with_policy([g for g in record[\'gt\'] if known&(1<<g[\'classe_id\'])],record[\'width\'],record[\'height\'],policy)\n    low=[p for p in pred_low if known&(1<<p[\'classe_id\'])]\n    all_pred=core.with_policy([p for p in pred_low if p[\'confianca\']>=conf],record[\'width\'],record[\'height\'],policy)\n    pred=[p for p in all_pred if known&(1<<p[\'classe_id\'])]\n    pairs,missing,extras=matcher(gt,pred)\n    events=[event(record,\'correspondencia\',gt[gi],pred[pi],float(iou),low,conf) for gi,pi,iou in pairs]\n    events += [event(record,\'objeto_nao_detectado\',gt[gi],None,None,low,conf) for gi in missing]\n    events += [event(record,\'previsao_nao_associada\',None,pred[pi],None,low,conf) for pi in extras]\n    messages=[policy.gerar_mensagem(p,\'medio\') for p in all_pred if core.alert_on(p)]\n    if any(not isinstance(s,str) or not s.strip() for s in messages):raise RuntimeError(\'Alerta sem mensagem.\')\n    im={\'coorte\':record[\'coorte\'],\'imagem\':record[\'name\'],\'id\':record[\'id\'],\'modelo\':\'B_e03\',\'limiar_confianca\':conf,\n        \'tp_alerta\':sum(e[\'ha_alerta_ref\'] and e[\'ha_alerta_pred\'] for e in events),\n        \'fn_alerta\':sum(e[\'ha_alerta_ref\'] and not e[\'ha_alerta_pred\'] for e in events),\n        \'fp_alerta\':sum(not e[\'ha_alerta_ref\'] and e[\'ha_alerta_pred\'] for e in events),\n        \'alta_ref\':sum(g[\'prioridade\']==\'alto\' for g in gt),\n        \'alta_sem_alerta\':sum(e[\'prioridade_ref\']==\'alto\' and not e[\'ha_alerta_pred\'] for e in events),\n        \'tp_detector\':len(pairs),\'fn_detector\':len(missing),\'fp_detector\':len(extras),\n        \'gt_avaliaveis\':len(gt),\'gt_ignorados\':len(record[\'gt\'])-len(gt),\n        \'predicoes_ignoradas_conf\':len(all_pred)-len(pred),\n        \'cobertura_completa_alertas\':known&core.ALERT_MASK==core.ALERT_MASK,\n        \'ha_alerta_ref\':any(core.alert_on(g) for g in gt),\'ha_alerta_pred\':any(core.alert_on(p) for p in pred),\n        \'mensagem_alerta\':\'\\n\'.join(messages)}\n    return im,events\n\ndef summarise(images,events,conf):\n    if not images:raise ValueError(\'Validação vazia.\')\n    total=lambda k:sum(r[k] for r in images)\n    complete=[r for r in images if r[\'cobertura_completa_alertas\']]\n    summary={\'modelo\':\'B_e03\',\'coorte\':images[0][\'coorte\'],\'imagens\':len(images),\'conf_alertas\':conf,\n        \'alertas_por_objeto\':core.alert_metrics(total(\'tp_alerta\'),total(\'fn_alerta\'),total(\'fp_alerta\')),\n        \'alta_ref\':total(\'alta_ref\'),\'alta_sem_alerta\':total(\'alta_sem_alerta\'),\n        \'taxa_alta_sem_alerta\':core.ratio(total(\'alta_sem_alerta\'),total(\'alta_ref\')),\n        \'detector_no_limiar\':core.alert_metrics(total(\'tp_detector\'),total(\'fn_detector\'),total(\'fp_detector\')),\n        \'gt_ignorados_sem_cobertura\':total(\'gt_ignorados\'),\'predicoes_ignoradas_conf\':total(\'predicoes_ignoradas_conf\'),\n        \'imagens_com_cobertura_completa_alertas\':len(complete),\n        \'alertas_por_imagem_cobertura_completa\':core.alert_metrics(\n            sum(r[\'ha_alerta_ref\'] and r[\'ha_alerta_pred\'] for r in complete),\n            sum(r[\'ha_alerta_ref\'] and not r[\'ha_alerta_pred\'] for r in complete),\n            sum(not r[\'ha_alerta_ref\'] and r[\'ha_alerta_pred\'] for r in complete)),\n        \'referencia\':\'Concordância com política V2 nas caixas; não valida risco físico.\',\n        \'nao_e_openimages_challenge\':True,\'ap_recalculada\':False}\n    classes=[]\n    for cid,name in enumerate(core.NAMES):\n        part=[e for e in events if e[\'classe_id\']==cid];n=sum(e[\'bbox_ref\']!=\'\' for e in part)\n        classes.append({\'classe_id\':cid,\'classe\':name,\'gt_avaliaveis\':n,\'suporte_reduzido\':n<30,\'conf_alertas\':conf,\n            **core.alert_metrics(sum(e[\'ha_alerta_ref\'] and e[\'ha_alerta_pred\'] for e in part),\n                                 sum(e[\'ha_alerta_ref\'] and not e[\'ha_alerta_pred\'] for e in part),\n                                 sum(not e[\'ha_alerta_ref\'] and e[\'ha_alerta_pred\'] for e in part)),\n            \'alta_ref\':sum(e[\'prioridade_ref\']==\'alto\' for e in part),\n            \'alta_sem_alerta\':sum(e[\'prioridade_ref\']==\'alto\' and not e[\'ha_alerta_pred\'] for e in part)})\n    return summary,classes\n\ndef evaluate_cache(records,cache,conf,policy,matcher,output=None):\n    if cache.keys()!={r[\'id\'] for r in records}:raise ValueError(\'Cache não corresponde aos IDs da validação.\')\n    images=[];events=[]\n    for r in records:\n        im,ev=analyse(r,cache[r[\'id\']],conf,policy,matcher);images.append(im);events.extend(ev)\n    summary,classes=summarise(images,events,conf)\n    if output is not None:\n        output=Path(output);output.mkdir(parents=True,exist_ok=True)\n        core.save_json(output/\'metricas.json\',summary)\n        core.save_csv(output/\'metricas_por_classe.csv\',classes)\n        core.save_csv(output/\'alertas_por_imagem.csv\',images)\n        core.save_csv(output/\'avaliacao_objetos.csv\',events)\n    return {\'summary\':summary,\'classes\':classes,\'images\':images,\'events\':events}\n\ndef parse_current_events(events):\n    result=[]\n    for r in events:\n        v=dict(r)\n        for key in [\'bbox_ref\',\'bbox_pred\']:v[key]=json.loads(v[key]) if v[key] else None\n        result.append(v)\n    return result\n\ndef check_reproduction(result,original,cohort):\n    old={r[\'id\']:r for r in original[\'images\']};new={r[\'id\']:r for r in result[\'images\']}\n    if old.keys()!=new.keys():raise ValueError(\'Reprodução exige os mesmos IDs.\')\n    rows=[]\n    fields=[\'tp_alerta\',\'fn_alerta\',\'fp_alerta\',\'alta_ref\',\'alta_sem_alerta\',\'tp_detector\',\'fn_detector\',\'fp_detector\',\'gt_avaliaveis\',\'gt_ignorados\',\'predicoes_ignoradas_conf_025\']\n    for iid in sorted(old):\n        for key in fields:\n            key_new=\'predicoes_ignoradas_conf\' if key==\'predicoes_ignoradas_conf_025\' else key\n            a,b=old[iid][key],new[iid][key_new]\n            rows.append({\'coorte\':cohort,\'id\':iid,\'metrica\':key,\'notebook12\':a,\'notebook14\':b,\'igual\':a==b})\n    old_gt=data.gt_map(original[\'events\']);new_gt=data.gt_map(parse_current_events(result[\'events\']))\n    if old_gt.keys()!=new_gt.keys():raise ValueError(\'Referências GT mudaram na reprodução.\')\n    object_rows=[]\n    for key,a in old_gt.items():\n        b=new_gt[key]\n        fields_gt=[\'evento\',\'prioridade_ref\',\'prioridade_pred\',\'ha_alerta_ref\',\'ha_alerta_pred\']\n        diffs=[field for field in fields_gt if a[field]!=b[field]]\n        object_rows.append({\'coorte\':cohort,\'id\':key[0],\'classe\':a[\'classe\'],\'bbox_ref\':json.dumps(key[2]),\n                            \'campos_diferentes\':\'|\'.join(diffs),\'igual\':not diffs})\n    return {\'coorte\':cohort,\'checagens_imagem\':len(rows),\'divergencias_imagem\':sum(not r[\'igual\'] for r in rows),\n            \'objetos_GT\':len(object_rows),\'divergencias_objetos\':sum(not r[\'igual\'] for r in object_rows),\n            \'reproduziu\':all(r[\'igual\'] for r in rows+object_rows)},rows,object_rows\n\ndef select_threshold(results,rule,reproduction_ok):\n    assessed=[]\n    for conf in THRESHOLDS:\n        home=results[\'home_val_historica\'][conf][\'summary\'];ext=results[\'openimages_val_nova\'][conf][\'summary\']\n        supported=all(s[\'alertas_por_objeto\'][k] is not None and math.isfinite(s[\'alertas_por_objeto\'][k])\n                      for s in [home,ext] for k in [\'precision\',\'recall\',\'f1\'])\n        if supported:\n            checks={**judge(home,ext,rule),\'metricas_com_suporte\':True}\n        else:\n            # Precisão sem nenhum alerta é indefinida; não vira zero ou aprovação.\n            checks={k:None for k in [\'home_precision_superior\',\'home_recall_preservado\',\'home_altas_preservadas\',\n                                    \'externa_precision_preservada\',\'externa_recall_preservado\',\'externa_altas_preservadas\']}\n            checks.update(elegivel=False,retencao_externa=False,metricas_com_suporte=False)\n        assessed.append({\'conf\':conf,\'home\':home,\'external\':ext,\'checks\':checks,\n                         \'comparavel_historico\':reproduction_ok,\'elegivel\':checks[\'elegivel\'] and reproduction_ok})\n    def ranking(r):\n        h=r[\'home\'][\'alertas_por_objeto\'];e=r[\'external\'][\'alertas_por_objeto\']\n        return (-h[\'precision\'],-h[\'recall\'],r[\'home\'][\'alta_sem_alerta\'],-e[\'f1\'],abs(r[\'conf\']-.25),-r[\'conf\'])\n    good=[r for r in assessed if r[\'elegivel\']]\n    chosen=min(good,key=ranking) if good else None\n    status=\'limiar_candidato_em_validacao\' if chosen else \'nenhum_limiar_elegivel\'\n    if not reproduction_ok:status=\'reproducao_divergente_sem_sugestao_de_limiar\'\n    return {\'status\':status,\'limiar_sugerido\':chosen[\'conf\'] if chosen else None,\n            \'comparabilidade_confirmada\':bool(reproduction_ok),\'assessed\':assessed,\n            \'limiar_galeria\':chosen[\'conf\'] if chosen else .10,\n            \'criterio_galeria_sem_elegivel\':\'0,10 fixado antes da inferência; somente diagnóstico.\',\n            \'modelo_em_uso_continua_baseline\':True,\'substituicao_automatica\':False}\n\ndef effects(results,bundle,cohort,cache):\n    old=data.gt_map(bundle[\'evaluations\'][\'candidato_10\'][cohort][\'events\'])\n    anchor=data.gt_map(parse_current_events(results[cohort][.25][\'events\']))\n    lost={k for k in anchor if old[k][\'ha_alerta_ref\'] and old[k][\'ha_alerta_pred\'] and not anchor[k][\'ha_alerta_pred\']}\n    changed=[];losses=[];priority=[]\n    for key in sorted(lost):\n        a,b=old[key],anchor[key]\n        if b[\'evento\']==\'correspondencia\':\n            priority.append({\'coorte\':cohort,\'id\':key[0],\'classe\':b[\'classe\'],\'bbox_ref\':json.dumps(key[2]),\n                \'prioridade_candidato10\':a[\'prioridade_pred\'],\'prioridade_B_025\':b[\'prioridade_pred\'],\n                \'confianca_candidato10\':a[\'confianca\'],\'confianca_B_025\':b[\'confianca\'],\n                \'observacao\':\'Objeto ainda associado; alerta pode mudar pela geometria/prioridade.\'})\n    for conf in THRESHOLDS:\n        current=data.gt_map(parse_current_events(results[cohort][conf][\'events\']))\n        for key,b in current.items():\n            a=anchor[key]\n            if a[\'ha_alerta_ref\'] and a[\'ha_alerta_pred\']!=b[\'ha_alerta_pred\']:\n                changed.append({\'coorte\':cohort,\'conf\':conf,\'id\':key[0],\'classe\':b[\'classe\'],\'bbox_ref\':json.dumps(key[2]),\n                    \'alerta_B_025\':a[\'ha_alerta_pred\'],\'alerta_no_limiar\':b[\'ha_alerta_pred\'],\n                    \'prioridade_ref\':b[\'prioridade_ref\'],\'confianca_associada\':b[\'confianca\']})\n            if key not in lost:continue\n            same=[p for p in cache[key[0]] if p[\'classe_id\']==key[1]]\n            ious=core.iou_matrix([{\'bbox\':list(key[2])}],same)[0]\n            best=max(range(len(same)),key=lambda i:ious[i]) if same else None\n            losses.append({\'coorte\':cohort,\'conf\':conf,\'id\':key[0],\'classe\':b[\'classe\'],\'bbox_ref\':json.dumps(key[2]),\n                \'prioridade_ref\':b[\'prioridade_ref\'],\'recuperado\':b[\'ha_alerta_pred\'],\n                \'evento_B_025\':a[\'evento\'],\'evento_no_limiar\':b[\'evento\'],\n                \'melhor_iou_cache_mesma_classe\':float(ious[best]) if best is not None else None,\n                \'confianca_dessa_proposta\':same[best][\'confianca\'] if best is not None else None,\n                \'limite\':\'Melhor IoU não garante associação nem alerta; coluna recuperado usa avaliação completa.\'})\n    return changed,losses,priority\n\n\ndef validate_predictions(pred,record):\n    """Valida estrutura; preserva coordenadas numéricas como no avaliador 11/12.\n\n    Área zero e limites da imagem são diagnósticos, não filtros do cache.\n    Nenhuma proposta é removida, recortada ou corrigida aqui.\n    """\n    totals={\'predicoes\':len(pred),\'area_zero\':0,\'fora_da_imagem\':0,\n            \'area_zero_na_grade\':0,\'fora_da_imagem_na_grade\':0,\'amostras\':[]}\n    for index,p in enumerate(pred):\n        reason=None\n        if not isinstance(p,dict):\n            raise ValueError(f"Previsão não é um dicionário: {record[\'id\']}; índice={index}; valor={p!r}")\n        cid=p.get(\'classe_id\');box=p.get(\'bbox\');score=p.get(\'confianca\')\n        if type(cid)!=int or not 0<=cid<len(core.NAMES):reason=\'classe_id inválida\'\n        elif p.get(\'classe\')!=core.NAMES[cid]:reason=\'nome da classe incompatível\'\n        elif type(score) not in (int,float) or not math.isfinite(score) or not .001-1e-8<=score<=1:\n            reason=\'confiança não finita ou fora da faixa de captura\'\n        elif not isinstance(box,(list,tuple)) or len(box)!=4:reason=\'bbox deve ter quatro coordenadas\'\n        elif not all(type(v) in (int,float) and math.isfinite(v) for v in box):reason=\'coordenada não numérica ou não finita\'\n        elif box[2]<box[0] or box[3]<box[1]:reason=\'coordenadas xyxy invertidas\'\n        if reason:\n            raise ValueError(f"Previsão incompatível com o cache: {record[\'id\']}; motivo={reason}; "\n                f"índice={index}; classe_id={cid!r}; confiança={score!r}; bbox={box!r}; "\n                f"dimensões=(w={record[\'width\']}, h={record[\'height\']})")\n        zero=box[2]==box[0] or box[3]==box[1]\n        outside=(min(box)<-.01 or box[2]>record[\'width\']+.01 or box[3]>record[\'height\']+.01)\n        totals[\'area_zero\']+=int(zero);totals[\'fora_da_imagem\']+=int(outside)\n        totals[\'area_zero_na_grade\']+=int(zero and score>=min(THRESHOLDS))\n        totals[\'fora_da_imagem_na_grade\']+=int(outside and score>=min(THRESHOLDS))\n        if (zero or outside) and len(totals[\'amostras\'])<5:\n            totals[\'amostras\'].append({\'indice\':index,\'classe\':p[\'classe\'],\'confianca\':score,\n                \'bbox\':list(box),\'area_zero\':zero,\'fora_da_imagem\':outside,\n                \'width\':record[\'width\'],\'height\':record[\'height\']})\n    return totals\n\ndef save_geometry_diagnostics(directory,items):\n    totals={key:sum(item[\'diagnostico\'][key] for item in items) for key in\n            [\'predicoes\',\'area_zero\',\'fora_da_imagem\',\'area_zero_na_grade\',\'fora_da_imagem_na_grade\']}\n    unusual=[item for item in items if item[\'diagnostico\'][\'area_zero\'] or item[\'diagnostico\'][\'fora_da_imagem\']]\n    report={\'correcao\':\'14-cache-validacao-1.0.1\',\'imagens\':len(items),\n        \'imagens_com_geometria_incomum\':len(unusual),\'totais\':totals,\n        \'propostas_removidas\':0,\'coordenadas_alteradas\':False,\n        \'observacao\':\'Geometria incomum preservada no cache e no protocolo; não constitui objeto aprovado. Reprodução em 0,25 continua obrigatória.\',\n        \'casos\':unusual}\n    core.save_json(Path(directory)/\'diagnostico_geometria_previsoes.json\',report)\n    return {k:v for k,v in report.items() if k!=\'casos\'}\n\ndef obtain_cache(model,records,directory,weight_sha,environment,device,predict_fn=None):\n    directory=Path(directory);directory.mkdir(parents=True,exist_ok=True)\n    path=directory/\'predicoes_conf_0p001.jsonl.gz\';meta_path=directory/\'cache_manifesto.json\'\n    request=cache_request(records,weight_sha,environment)\n    if path.is_file() and meta_path.is_file():\n        meta=json.loads(meta_path.read_text())\n        if meta[\'solicitacao\']!=request or meta[\'cache_sha256\']!=core.sha256_file(path):\n            raise RuntimeError(\'Cache existente não corresponde aos pesos/entradas/ambiente. Não será reutilizado.\')\n        cached=read_cache(path,records)\n        diagnostics=[{\'id\':r[\'id\'],\'diagnostico\':validate_predictions(cached[r[\'id\']],r)} for r in records]\n        geometry=save_geometry_diagnostics(directory,diagnostics)\n        print(\'Cache conferido e reutilizado:\',records[0][\'coorte\'],flush=True)\n        return path,{**meta,\'geometria_resumo\':geometry,\'reutilizado_nesta_execucao\':True}\n    predict=predict_fn or core.model_predictions\n    tmp=path.with_name(path.name+\'.tmp\')\n    saturated=0;diagnostics=[]\n    with gzip.open(tmp,\'wt\',encoding=\'utf-8\',compresslevel=4) as f:\n        for i,r in enumerate(records,1):\n            if r[\'role\'] not in {\'home_val\',\'val_nova\'}:raise ValueError(\'Papel proibido na inferência.\')\n            pred=predict(model,r,device)\n            try:diagnostic=validate_predictions(pred,r)\n            except ValueError as exc:\n                core.save_json(directory/\'previsao_rejeitada.json\',{\'id\':r[\'id\'],\'mensagem\':str(exc),\n                    \'width\':r[\'width\'],\'height\':r[\'height\'],\'propostas\':pred})\n                raise\n            diagnostics.append({\'id\':r[\'id\'],\'diagnostico\':diagnostic})\n            if len(pred)==PREDICT_ARGS[\'max_det\']:saturated+=1\n            f.write(json.dumps({\'id\':r[\'id\'],\'imagem\':r[\'name\'],\'image_sha256\':r[\'sha256\'],\n                                \'label_sha256\':r[\'label_sha256\'],\'predicoes\':pred},ensure_ascii=False)+\'\\n\')\n            if i%100==0 or i==len(records):print(f"B_e03 / {r[\'coorte\']}: {i}/{len(records)}",flush=True)\n    tmp.replace(path)\n    geometry=save_geometry_diagnostics(directory,diagnostics)\n    meta={\'solicitacao\':request,\'cache_sha256\':core.sha256_file(path),\'concluido\':True,\n          \'validacao_previsoes\':\'14-cache-validacao-1.0.1\',\'geometria_resumo\':geometry,\n          \'imagens_atingiram_max_det\':saturated,\'limite\':\'Propostas pós-NMS em confiança 0,001; max_det=300, sem acesso aos logits brutos.\'}\n    core.save_json(meta_path,meta)\n    read_cache(path,records)\n    print(\'Diagnóstico geométrico do cache:\',geometry[\'totais\'],flush=True)\n    return path,{**meta,\'reutilizado_nesta_execucao\':False}\n', 'conf14_reports.py': '"""Relatórios automáticos e galeria do diagnóstico de confiança."""\nfrom __future__ import annotations\nimport html\nimport json\nfrom pathlib import Path\nimport zipfile\nimport matplotlib\nmatplotlib.use(\'Agg\')\nimport matplotlib.pyplot as plt\nfrom matplotlib.ticker import MaxNLocator\nfrom PIL import Image\nimport diag11_core as core\nimport diag11_reports as r11\nimport conf14_evaluation as evaluation\nimport conf14_data as data\n\nLABELS={\'home_val_historica\':\'Home VAL histórica\',\'openimages_val_nova\':\'Open Images VAL de desenvolvimento\'}\n\ndef pct(value):return \'—\' if value is None else f\'{100*value:.2f}%\'\n\ndef comparison_rows(results,bundle,selection):\n    rows=[]\n    for cohort in data.COHORTS:\n        anchor=results[cohort][.25][\'summary\'];a=anchor[\'alertas_por_objeto\']\n        for model in [\'baseline\',\'candidato_10\']:\n            s=bundle[\'evaluations\'][model][cohort][\'summary\'];m=s[\'alertas_por_objeto\']\n            rows.append({\'coorte\':cohort,\'modelo\':model,\'conf\':.25,\'origem\':\'registro autenticado do notebook 12\',\n                \'imagens\':s[\'imagens\'],**m,\'alta_ref\':s[\'alta_ref\'],\'alta_sem_alerta\':s[\'alta_sem_alerta\'],\n                \'delta_fp_vs_B_025\':m[\'fp\']-a[\'fp\'],\'delta_fn_vs_B_025\':m[\'fn\']-a[\'fn\'],\n                \'elegivel_14\':False})\n        for entry in selection[\'assessed\']:\n            conf=entry[\'conf\'];s=results[cohort][conf][\'summary\'];m=s[\'alertas_por_objeto\']\n            rows.append({\'coorte\':cohort,\'modelo\':\'B_e03\',\'conf\':conf,\'origem\':\'cache da nova inferência do notebook 14\',\n                \'imagens\':s[\'imagens\'],**m,\'alta_ref\':s[\'alta_ref\'],\'alta_sem_alerta\':s[\'alta_sem_alerta\'],\n                \'delta_fp_vs_B_025\':m[\'fp\']-a[\'fp\'],\'delta_fn_vs_B_025\':m[\'fn\']-a[\'fn\'],\n                \'elegivel_14\':entry[\'elegivel\']})\n    return rows\n\ndef plot_curves(output,results,bundle):\n    fig,axes=plt.subplots(4,2,figsize=(12,12),sharex=True,layout=\'constrained\')\n    for col,cohort in enumerate(data.COHORTS):\n        for row,key in enumerate([\'precision\',\'recall\',\'fp\',\'alta_sem_alerta\']):\n            values=[]\n            for conf in evaluation.THRESHOLDS:\n                s=results[cohort][conf][\'summary\']\n                v=s[key] if key==\'alta_sem_alerta\' else s[\'alertas_por_objeto\'][key]\n                values.append(100*v if row<2 and v is not None else v)\n            ax=axes[row,col];ax.plot(evaluation.THRESHOLDS,values,\'o-\',color=\'#176ca4\',label=\'B_e03\')\n            for model,color in [(\'baseline\',\'#6c7785\'),(\'candidato_10\',\'#d17920\')]:\n                s=bundle[\'evaluations\'][model][cohort][\'summary\']\n                v=s[key] if key==\'alta_sem_alerta\' else s[\'alertas_por_objeto\'][key]\n                if v is not None:ax.axhline(100*v if row<2 else v,color=color,linestyle=\'--\',label=model+\' / 0,25\')\n            ax.axvline(.25,color=\'#97a1ad\',linewidth=1,alpha=.6)\n            ax.grid(alpha=.2)\n            ax.set_ylabel([\'Precisão de alertas (%)\',\'Recall de alertas (%)\',\'Falsos alertas avaliáveis\',\'Objetos de prioridade alta sem alerta\'][row])\n            if row<2:ax.set_ylim(0,100)\n            else:\n                ax.set_ylim(bottom=0);ax.yaxis.set_major_locator(MaxNLocator(integer=True))\n        axes[0,col].set_title(LABELS[cohort]);axes[0,col].legend(fontsize=8)\n        axes[3,col].set_xticks(evaluation.THRESHOLDS)\n        axes[3,col].set_xlabel(\'Limiar global de confiança\')\n    fig.suptitle(\'Pesos B_e03 fixos · política V2 fixa · uma inferência por imagem\',fontsize=13)\n    fig.savefig(Path(output)/\'curvas_confianca.png\',dpi=150)\n    plt.close(fig)\n\ndef gallery(output,cohorts,caches,results,focus,policy,seed=42):\n    output=Path(output);index=[];cards=[]\n    for cohort,records in cohorts.items():\n        chosen=r11.select_gallery_cases(results[cohort][.25][\'images\'],results[cohort][focus][\'images\'],limit=12,seed=seed)\n        mapping={r[\'id\']:r for r in records}\n        for i,item in enumerate(chosen):\n            r=mapping[item[\'id\']];pred=caches[cohort][r[\'id\']]\n            gt=core.with_policy([g for g in r[\'gt\'] if r[\'known_mask\']&(1<<g[\'classe_id\'])],r[\'width\'],r[\'height\'],policy)\n            original=core.with_policy([p for p in pred if p[\'confianca\']>=.25],r[\'width\'],r[\'height\'],policy)\n            changed=core.with_policy([p for p in pred if p[\'confianca\']>=focus],r[\'width\'],r[\'height\'],policy)\n            panels=[r11.draw_panel(r,gt,\'Referência V2 avaliável\'),\n                    r11.draw_panel(r,original,\'B_e03 / confiança 0,25\'),\n                    r11.draw_panel(r,changed,f\'B_e03 / confiança {focus:.2f}\')]\n            canvas=Image.new(\'RGB\',(1620,470),\'#101a28\')\n            for j,panel in enumerate(panels):canvas.paste(panel,(j*540,0))\n            name=f\'galeria/{cohort}_{i:02}.jpg\';path=output/name;path.parent.mkdir(parents=True,exist_ok=True)\n            canvas.save(path,quality=86)\n            index.append({**item,\'coorte\':cohort,\'conf_comparada\':focus,\'arquivo\':name,\n                          \'creditos\':r.get(\'creditos\',{}),\'preenchimento_manual\':False})\n            cards.append(\'<article><h3>\'+html.escape(LABELS[cohort]+\' · \'+item[\'categoria\'])+\'</h3><img loading="lazy" src="\'+name+\'" alt="Referência e previsões do mesmo modelo em dois limiares"><p>\'+html.escape(r[\'name\'])+\'</p></article>\')\n    core.save_json(output/\'indice_galeria.json\',index)\n    return \'\'.join(cards)\n\ndef selection_table(selection):\n    head=[\'Confiança\',\'P Home\',\'R Home\',\'Alta omitida Home\',\'P externa\',\'R externa\',\'Alta omitida externa\',\'Cumpre todos os requisitos\']\n    table=\'<table><thead><tr>\'+\'\'.join(\'<th>\'+v+\'</th>\' for v in head)+\'</tr></thead><tbody>\'\n    for entry in selection[\'assessed\']:\n        h,e=entry[\'home\'],entry[\'external\'];ha,ea=h[\'alertas_por_objeto\'],e[\'alertas_por_objeto\']\n        values=[f"{entry[\'conf\']:.2f}",pct(ha[\'precision\']),pct(ha[\'recall\']),h[\'alta_sem_alerta\'],pct(ea[\'precision\']),pct(ea[\'recall\']),e[\'alta_sem_alerta\'],\'Sim, nas validações\' if entry[\'elegivel\'] else \'Não\']\n        table+=\'<tr>\'+\'\'.join(\'<td>\'+html.escape(str(v))+\'</td>\' for v in values)+\'</tr>\'\n    return table+\'</tbody></table>\'\n\ndef build_report(output,results,bundle,selection,reproduction,cohorts,caches,policy,config):\n    output=Path(output)\n    rows=comparison_rows(results,bundle,selection)\n    core.save_csv(output/\'comparacao_confianca.csv\',rows)\n    core.save_json(output/\'selecao_limiar.json\',selection)\n    checks=[{\'conf\':e[\'conf\'],**e[\'checks\'],\'comparavel_historico\':e[\'comparavel_historico\'],\n             \'elegivel_final\':e[\'elegivel\']} for e in selection[\'assessed\']]\n    core.save_csv(output/\'requisitos_por_limiar.csv\',checks)\n    changed=[];losses=[];priority=[]\n    for cohort in data.COHORTS:\n        a,b,c=evaluation.effects(results,bundle,cohort,caches[cohort]);changed+=a;losses+=b;priority+=c\n    core.save_csv(output/\'alertas_que_mudaram_vs_B_025.csv\',changed,\n        [\'coorte\',\'conf\',\'id\',\'classe\',\'bbox_ref\',\'alerta_B_025\',\'alerta_no_limiar\',\'prioridade_ref\',\'confianca_associada\'])\n    core.save_csv(output/\'perdas_vs_candidato10_por_limiar.csv\',losses,\n        [\'coorte\',\'conf\',\'id\',\'classe\',\'bbox_ref\',\'prioridade_ref\',\'recuperado\',\'evento_B_025\',\'evento_no_limiar\',\'melhor_iou_cache_mesma_classe\',\'confianca_dessa_proposta\',\'limite\'])\n    core.save_csv(output/\'perdas_com_objeto_ainda_detectado.csv\',priority,\n        [\'coorte\',\'id\',\'classe\',\'bbox_ref\',\'prioridade_candidato10\',\'prioridade_B_025\',\'confianca_candidato10\',\'confianca_B_025\',\'observacao\'])\n    recovery=[]\n    for cohort in data.COHORTS:\n        for conf in evaluation.THRESHOLDS:\n            part=[r for r in losses if r[\'coorte\']==cohort and r[\'conf\']==conf]\n            high=[r for r in part if r[\'prioridade_ref\']==\'alto\']\n            recovery.append({\'coorte\':cohort,\'conf\':conf,\'perdas_B_025_vs_candidato10\':len(part),\n                \'dessas_perdas_recuperadas\':sum(r[\'recuperado\'] for r in part),\n                \'perdas_altas_B_025_vs_candidato10\':len(high),\n                \'dessas_perdas_altas_recuperadas\':sum(r[\'recuperado\'] for r in high)})\n    core.save_csv(output/\'recuperacao_perdas_resumo.csv\',recovery)\n    plot_curves(output,results,bundle)\n    cards=gallery(output,cohorts,caches,results,selection[\'limiar_galeria\'],policy)\n    if not selection[\'comparabilidade_confirmada\']:\n        status=\'As contagens/associações em 0,25 divergiram do experimento 12. Os resultados são diagnósticos e nenhum limiar é sugerido.\'\n    elif selection[\'limiar_sugerido\'] is None:\n        status=\'Nenhum limiar da grade cumpriu todos os requisitos. O baseline continua em uso.\'\n    else:\n        status=f"O limiar {selection[\'limiar_sugerido\']:.2f} cumpriu os requisitos nas validações de desenvolvimento. É uma configuração candidata, sem substituição automática do baseline."\n    p=config[\'protocolo\'][\'regra_original_12\'];h=p[\'home\'];e=p[\'externa\']\n    refs=\'<table><tr><th>Validação</th><th>Referência em 0,25</th><th>Precisão</th><th>Recall</th><th>Alta sem alerta</th></tr>\'\n    for cohort in data.COHORTS:\n        for model in [\'baseline\',\'candidato_10\']:\n            s=bundle[\'evaluations\'][model][cohort][\'summary\'];a=s[\'alertas_por_objeto\']\n            refs+=\'<tr>\'+\'\'.join(\'<td>\'+html.escape(str(v))+\'</td>\' for v in [LABELS[cohort],model,pct(a[\'precision\']),pct(a[\'recall\']),s[\'alta_sem_alerta\']])+\'</tr>\'\n    refs+=\'</table>\'\n    reproduction_table=\'<table><tr><th>Validação</th><th>Divergências por imagem</th><th>Divergências por objeto</th><th>Reprodução confirmada</th></tr>\'\n    for r in reproduction:\n        reproduction_table+=\'<tr>\'+\'\'.join(\'<td>\'+html.escape(str(v))+\'</td>\' for v in [LABELS[r[\'coorte\']],r[\'divergencias_imagem\'],r[\'divergencias_objetos\'],\'Sim\' if r[\'reproduziu\'] else \'Não\'])+\'</tr>\'\n    reproduction_table+=\'</table>\'\n    css=\'body{font:16px/1.6 system-ui;background:#f3f6fa;color:#182638;margin:0}main{max-width:1320px;margin:auto;padding:28px}h1{line-height:1.2}h2{margin-top:30px}article,.note{background:white;border:1px solid #d6e0eb;border-radius:10px;padding:18px;margin:18px 0}table{border-collapse:collapse;background:white;width:100%;font-size:14px}th,td{border:1px solid #d6e0eb;padding:9px;text-align:left}.scroll{overflow:auto}img{width:100%;height:auto}a{color:#12689c}code{overflow-wrap:anywhere}\'\n    page=\'<!doctype html><html lang="pt-BR"><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1"><title>TCC — Diagnóstico 14</title><style>\'+css+\'</style><main><h1>14 — Diagnóstico automático de confiança</h1>\'\n    page+=\'<div class="note"><strong>\'+html.escape(status)+\'</strong><p>Pesos B_e03 e política V2 fixos; zero novos treinos. Sem formulários. As referências do baseline e do candidato 10 são os registros autenticados do notebook 12, sem nova inferência desses dois modelos.</p></div>\'\n    page+=\'<h2>Grade fixa e requisitos preservados</h2><p>Um único limiar global deve funcionar nas duas validações. Home exige precisão superior a \'+pct(h[\'precision_maior_que\'])+\', recall de pelo menos \'+pct(h[\'recall_minimo\'])+\' e no máximo \'+str(h[\'alta_sem_alerta_maximo\'])+\' omissões altas. A externa exige precisão de pelo menos \'+pct(e[\'precision_minima\'])+\', recall de pelo menos \'+pct(e[\'recall_minimo\'])+\' e no máximo \'+str(e[\'alta_sem_alerta_maximo\'])+\' omissões altas. O JSON conserva os valores exatos.</p>\'\n    page+=\'<div class="scroll">\'+selection_table(selection)+\'</div><h2>Troca entre recuperação e falsos alertas</h2><img src="curvas_confianca.png" alt="Precisão, recall, falsos alertas e omissões altas em função do limiar">\'\n    page+=\'<p><a href="comparacao_confianca.csv">Comparação completa</a> · <a href="recuperacao_perdas_resumo.csv">Recuperação das perdas anteriores</a> · <a href="requisitos_por_limiar.csv">Requisitos por limiar</a></p>\'\n    page+=\'<p>As propostas numéricas do modelo foram preservadas, inclusive caixas de área zero. Os arquivos caches/&lt;coorte&gt;/diagnostico_geometria_previsoes.json registram geometrias incomuns. Não houve recorte ou remoção automática de propostas.</p>\'\n    page+=\'<h2>Reprodução em 0,25</h2><div class="scroll">\'+reproduction_table+\'</div><p>A confirmação exige igualdade dos contadores por imagem e dos estados por objeto. Qualquer divergência impede sugerir limiar por esses requisitos históricos; as diferenças ficam nos CSVs de reprodução.</p>\'\n    page+=\'<h2>Referências autenticadas do experimento 12</h2><div class="scroll">\'+refs+\'</div>\'\n    page+=\'<h2>Casos selecionados automaticamente</h2><p>Referência, B em 0,25 e B no limiar \'+f"{selection[\'limiar_galeria\']:.2f}"+\'. Sem limiar elegível, o painel usa 0,10, definido previamente apenas para diagnóstico. A galeria dirigida não estima prevalência de erros e não pede preenchimento.</p>\'+cards\n    page+=\'<h2>Limites</h2><p>Esta é seleção de ponto de operação em validações já usadas no desenvolvimento, não teste final independente nem prova de calibração probabilística. A origem histórica do treino do baseline permanece não determinada. Open Images tem cobertura parcial por imagem/classe; somente classes avaliáveis entram nas métricas. Confiança menor pode aumentar falsos alertas e alterar associações. As previsões são pós-NMS com max_det=300. A política V2 nas caixas não constitui validação independente de risco físico. As 300 imagens reservadas e os testes ficaram sem extração/inferência nesta etapa. Os rótulos não foram modificados; a auditoria anterior por IA não é uma aprovação humana da base.</p></main></html>\'\n    (output/\'relatorio_diagnostico_14.html\').write_text(page,encoding=\'utf-8\')\n    summary={\'versao\':evaluation.VERSION,\'concluido\':True,\'status\':selection[\'status\'],\'mensagem\':status,\n        \'limiar_sugerido\':selection[\'limiar_sugerido\'],\'grade\':list(evaluation.THRESHOLDS),\n        \'reproducao_confirmada\':selection[\'comparabilidade_confirmada\'],\n        \'modelo_inferido\':\'B_e03\',\'imagens_avaliadas\':sum(len(v) for v in cohorts.values()),\n        \'imagens_inferidas_nesta_execucao\':config.get(\'imagens_inferidas_nesta_execucao\'),\n        \'novos_treinamentos\':0,\'preenchimento_manual_obrigatorio\':False,\n        \'baseline_e_candidato10_reinferidos\':False,\'referencias\':\'pacote 12 autenticado\',\n        \'modelo_em_uso_continua_baseline\':True,\'modelos_substituidos\':False,\n        \'reserva_futura_extraida\':False,\'reserva_futura_inferida\':False,\n        \'test_home_inferido\':False,\'test_oi20k_inferido\':False,\'rotulos_alterados\':False,\n        \'validacoes_usadas_na_selecao\':True,\'nao_e_teste_final_independente\':True,\n        \'correcao_cache\':\'14-cache-validacao-1.0.1\',\n        \'geometria_previsoes\':{c:m.get(\'geometria_resumo\',{}) for c,m in config.get(\'caches\',{}).items()},\n        \'peso_B_e03_sha256\':config[\'peso_B_e03_sha256\'],\'dataset_sha256\':config[\'dataset_sha256\'],\n        \'recuperacao_perdas\':recovery,\n        \'proximo_passo\':(\'Congelar a configuração candidata e preparar avaliação futura independente, sem uso automático da reserva.\' if selection[\'limiar_sugerido\'] is not None else\n                        \'Analisar as diferenças de reprodução antes de selecionar configuração.\' if not selection[\'comparabilidade_confirmada\'] else\n                        \'Encerrar esta hipótese de limiar; planejar revisão automática mais ampla dos dados e novo experimento documentado.\')}\n    core.save_json(output/\'resumo_diagnostico_14.json\',summary)\n    (output/\'LEIA_RESULTADO.md\').write_text(\'# Diagnóstico 14\\n\\n\'+status+\'\\n\\nAbra relatorio_diagnostico_14.html. As comparações e a galeria já foram geradas; não há fichas para preencher. Resultados em pacote_diagnostico_14.zip para documentação.\\n\\nO pacote inclui as previsões completas pós-NMS em confiança 0,001, para permitir novas análises sem repetir a inferência. Nenhum treinamento foi executado ou modelo em uso foi substituído.\\n\',encoding=\'utf-8\')\n    return summary\n\ndef package_reports(output):\n    output=Path(output);archive=output/\'pacote_diagnostico_14.zip\'\n    with zipfile.ZipFile(archive,\'w\',compression=zipfile.ZIP_DEFLATED,compresslevel=4) as z:\n        for path in sorted(output.rglob(\'*\')):\n            if path.is_file() and path!=archive and not path.name.endswith(\'.tmp\') and path.suffix not in {\'.pt\',\'.pkl\'}:\n                z.write(path,path.relative_to(output))\n    return archive\n'}

PASTA_CODIGO=PASTA_LOCAL/'codigo'
PASTA_CODIGO.mkdir(exist_ok=True)
for nome,fonte in MODULOS_EMBUTIDOS.items():
    (PASTA_CODIGO/nome).write_text(fonte,encoding='utf-8')
for nome,digest in EXPECTED['codigo_11'].items():
    if hashlib.sha256(MODULOS_EMBUTIDOS[nome].encode()).hexdigest()!=digest:
        raise RuntimeError('Código original do 11 alterado: '+nome)
for nome in ['exp12_data.py','exp12_protocol.py']:
    if hashlib.sha256(MODULOS_EMBUTIDOS[nome].encode()).hexdigest()!=EXPECTED['codigo_12'][nome]:
        raise RuntimeError('Código original do 12 alterado: '+nome)
sys.path.insert(0,str(PASTA_CODIGO))
import diag11_core as core
import exp12_data as dados12
import conf14_data as dados14
import conf14_evaluation as avaliacao14
import conf14_reports as relatorios14
if avaliacao14.THRESHOLDS!=(.05,.10,.15,.20,.25,.30,.40):
    raise RuntimeError('Grade alterada. Documente outro experimento para mudar os limiares.')
for nome,fonte in MODULOS_EMBUTIDOS.items():
    p=PASTA_RESULTADOS/'codigo'/nome;p.parent.mkdir(exist_ok=True);p.write_text(fonte,encoding='utf-8')
core.save_json(PASTA_RESULTADOS/'referencia_auditoria_13_automatica.json',AUDITORIA_13_AUTOMATICA)
print('Avaliador carregado. A revisão humana não é condição desta execução.')


## 3. Conectar ao Drive e autenticar as entradas

A conexão reutiliza o Secret dos notebooks anteriores. O peso B só será
carregado depois da verificação SHA256. Os pacotes 11 e 12 fornecem as
identidades das validações, as referências e os requisitos já definidos.


In [ ]:
from pathlib import Path
import base64
import configparser
import json
import shutil
import subprocess
import urllib.request
import zipfile
from kaggle_secrets import UserSecretsClient

SECRET_DRIVE = "GDRIVE_RCLONE_CONFIG_B64"
REMOTE_DRIVE = "gdrive_tcc"
ORIGEM_DRIVE = f"{REMOTE_DRIVE}:TCC_Visao_Computacional"
DRIVE_TCC = Path("/tmp/tcc_100k_entradas")
ENVIAR_RESULTADOS_AO_DRIVE = True
EXECUTAR_CALIBRACAO = False

# Credenciais ficam fora da pasta de resultados salva pelo Kaggle.
PASTA_RCLONE = Path("/tmp/tcc_rclone")
PASTA_RCLONE.mkdir(mode=0o700, parents=True, exist_ok=True)
PASTA_RCLONE.chmod(0o700)
CONFIG_RCLONE = PASTA_RCLONE / "gdrive_tcc.conf"
RCLONE = PASTA_RCLONE / "rclone"

try:
    segredo = UserSecretsClient().get_secret(SECRET_DRIVE)
except Exception:
    raise RuntimeError(
        f"Crie o Secret {SECRET_DRIVE} em Add-ons → Secrets e habilite "
        "o acesso deste notebook. Siga a preparação descrita acima."
    ) from None
try:
    configuracao_bytes = base64.b64decode(segredo.strip(), validate=True)
    configuracao = configparser.ConfigParser(interpolation=None)
    configuracao.read_string(configuracao_bytes.decode("utf-8-sig"))
except Exception:
    raise RuntimeError(
        "O Secret não contém a configuração rclone codificada corretamente. "
        "Copie novamente pelo comando PowerShell e atualize o Secret."
    ) from None
if (not configuracao.has_section(REMOTE_DRIVE)
        or configuracao.get(REMOTE_DRIVE, "type", fallback="") != "drive"
        or not configuracao.get(REMOTE_DRIVE, "token", fallback="")
        or not configuracao.get(REMOTE_DRIVE, "client_id", fallback="")):
    raise RuntimeError(
        "A configuração precisa da conexão gdrive_tcc, tipo drive, "
        "cliente OAuth próprio e autorização concluída no navegador."
    )
if (ENVIAR_RESULTADOS_AO_DRIVE
        and configuracao.get(REMOTE_DRIVE, "scope", fallback="") != "drive"):
    raise RuntimeError(
        "Para salvar os resultados, configure scope 1 (drive, leitura e "
        "gravação) no rclone e atualize o Secret."
    )
# Esta conexão usa Meu Drive, não um Drive compartilhado.
configuracao.remove_option(REMOTE_DRIVE, "team_drive")
with CONFIG_RCLONE.open("w", encoding="utf-8") as arquivo_config:
    configuracao.write(arquivo_config)
CONFIG_RCLONE.chmod(0o600)
del segredo, configuracao_bytes, configuracao

if not RCLONE.is_file():
    import os

    executavel = shutil.which("rclone")
    if executavel is None:
        prefixo = [] if os.geteuid() == 0 else ["sudo"]
        subprocess.run(prefixo + ["apt-get", "update", "-qq"], check=True)
        subprocess.run(
            prefixo + ["apt-get", "install", "-y", "--no-install-recommends", "rclone"],
            check=True,
        )
        executavel = shutil.which("rclone")

    if executavel is None:
        raise RuntimeError("A instalação terminou, mas o executável rclone não foi localizado.")

    shutil.copy2(executavel, RCLONE)
    RCLONE.chmod(0o700)
    print("rclone instalado pelo gerenciador de pacotes do Linux.")

DRIVE_TCC.mkdir(parents=True, exist_ok=True)
if shutil.disk_usage(DRIVE_TCC).free < 10 * 1024**3:
    raise RuntimeError("Reserve pelo menos 10 GiB livres para baixar as entradas.")

def baixar_arquivo_drive(relativo, destino):
    destino = Path(destino)
    destino.parent.mkdir(parents=True, exist_ok=True)
    print("Baixando:", relativo, flush=True)
    concluido = subprocess.run(
        [str(RCLONE), "--config", str(CONFIG_RCLONE), "copyto",
         f"{ORIGEM_DRIVE}/{relativo}", str(destino),
         "--retries", "3", "--low-level-retries", "5"], check=False,
    )
    if concluido.returncode != 0 or not destino.is_file():
        raise RuntimeError(f"Falha ao baixar {relativo}. Confira a saída do rclone, a autorização e a Internet.")
    return destino


def enviar_resultados_drive(etapa="manual", obrigatorio=False):
    if not ENVIAR_RESULTADOS_AO_DRIVE:
        return False
    pasta = Path(PASTA_RESULTADOS)
    destino = f"{ORIGEM_DRIVE}/resultados/{NOME_EXPERIMENTO}"
    print(f"Enviando resultados ao Drive ({etapa})...", flush=True)
    try:
        enviado = subprocess.run(
            [str(RCLONE), "--config", str(CONFIG_RCLONE), "copy",
             str(pasta), destino, "--exclude", "**/*.tmp",
             "--retries", "2", "--low-level-retries", "3",
             "--contimeout", "15s", "--timeout", "60s"],
            check=False,
        )
        if enviado.returncode != 0:
            raise RuntimeError(f"rclone retornou código {enviado.returncode}")
    except (OSError, RuntimeError) as exc:
        mensagem = (
            f"Envio ao Drive falhou em {etapa}: {exc}. "
            "Os arquivos locais continuam em /kaggle/working/resultados_tcc. "
            "Confira a autorização e execute a última etapa para repetir o envio."
        )
        if obrigatorio:
            raise RuntimeError(mensagem) from None
        print(mensagem, flush=True)
        return False
    print("Resultados copiados ao Drive.", flush=True)
    return True

def enviar_resultados(etapa,obrigatorio=True):
    return enviar_resultados_drive(etapa,obrigatorio=obrigatorio)

def registrar_falha(exc,etapa):
    core.save_json(PASTA_RESULTADOS/'execucao_interrompida.json',{
        'concluido':False,'etapa':etapa,'tipo':type(exc).__name__,'mensagem':str(exc)[:1200],
        'novos_treinamentos':0,'preenchimento_manual_obrigatorio':False,
        'modelos_substituidos':False})
    pacote=relatorios14.package_reports(PASTA_RESULTADOS)
    try:enviar_resultados('execução interrompida: '+etapa,obrigatorio=False)
    except Exception:pass
    print('Pacote de diagnóstico parcial:',pacote,flush=True)
    display(FileLink(str(pacote)))

ENTRADAS=PASTA_LOCAL/'entradas'
try:
    PACOTE_12=baixar_arquivo_drive(PASTA_EXECUCAO_12+'/pacote_experimento_12.zip',ENTRADAS/'pacote_12.zip')
    BUNDLE=dados14.load_source12(PACOTE_12,EXPECTED,PASTA_RESULTADOS)
    PACOTE_11=baixar_arquivo_drive(PASTA_EXECUCAO_11+'/pacote_diagnostico_11.zip',ENTRADAS/'pacote_11.zip')
    RESUMO_11,CONFIG_11,LOCAL_11,MANIFESTO_NOVO=dados12.read_package11(
        PACOTE_11,EXPECTED['pacote_11'],{'antigo':EXPECTED['baseline'],'candidato':EXPECTED['candidato']},EXPECTED['dataset_novo'])
    if LOCAL_11!=LOCALIZACAO_11 or CONFIG_11['politica_sha256']!=EXPECTED['politica']:
        raise RuntimeError('Identidade da validação ou política incompatível.')
    ZIP_HOME=baixar_arquivo_drive('datasets/homeobjects_sunrgbd_validado.zip',ENTRADAS/'home.zip')
    ZIP_NOVO=baixar_arquivo_drive(LOCALIZACAO_11['pasta_drive']+'/'+LOCALIZACAO_11['arquivo'],ENTRADAS/'validacao_11.zip')
    PESO_B=baixar_arquivo_drive(PASTA_EXECUCAO_12+'/pesos_checkpoints/B_home_replay_e03.pt',ENTRADAS/'B_home_replay_e03.pt')
    if core.sha256_file(PESO_B)!=EXPECTED['B_e03']:
        raise RuntimeError('Peso B_e03 diferente do checkpoint avaliado no 12.')
    FROZEN_FILES=[(p,core.sha256_file(p)) for p in [PACOTE_12,PACOTE_11,ZIP_HOME,ZIP_NOVO,PESO_B]]
    print('Pacotes e pesos autenticados. Nenhum lote de treino será recuperado.')
except Exception as exc:
    registrar_falha(exc,'entradas')
    raise


## 4. Extrair somente as duas validações

Em Home, apenas as 456 imagens VAL e seus labels serão extraídos.
Os hashes TRAIN/VAL são conferidos no manifesto, sem abrir os JPEGs de treino.
Na externa, o extrator original preserva as 300 reservadas dentro do ZIP.
As referências de caixas avaliáveis devem coincidir com as do experimento 12.


In [ ]:
try:
    HOME_VAL,HASHES_HOME,congelados,AUDITORIA_HOME=dados14.extract_home_val(
        ZIP_HOME,PASTA_LOCAL/'home',EXPECTED['home_zip'],EXPECTED['test_home'],EXPECTED['politica'])
    FROZEN_FILES+=congelados
    sys.path.insert(0,str(PASTA_LOCAL/'home/codigo'))
    import src.politica_alertas as politica
    import src.validacao_alertas as avaliacao_base
    if (core.sha256_file(politica.__file__)!=EXPECTED['politica'] or
        float(avaliacao_base.IOU_CORRESPONDENCIA)!=.5 or float(avaliacao_base.IOU_NMS)!=.7 or int(avaliacao_base.IMGSZ)!=640):
        raise RuntimeError('Política ou parâmetros do associador incompatíveis.')
    VAL_NOVA,IDS_RESERVA,congelados=dados12.load_frozen_validation(
        ZIP_NOVO,PASTA_LOCAL/'validacao',EXPECTED['zip_novo'],EXPECTED['dataset_novo'],MANIFESTO_NOVO)
    FROZEN_FILES+=congelados
    if len(IDS_RESERVA)!=300 or any('reserva_futura' in p.parts for p in (PASTA_LOCAL/'validacao').rglob('*')):
        raise RuntimeError('Reserva extraída ou quantidade de IDs incompatível.')
    if {r['sha256'] for r in MANIFESTO_NOVO['arquivos']}&HASHES_HOME:
        raise RuntimeError('Coincidência de bytes entre a base externa e Home.')
    COORTES={'home_val_historica':HOME_VAL,'openimages_val_nova':VAL_NOVA}
    dados14.validate_cohorts(COORTES,BUNDLE,IDS_RESERVA)
    AUDITORIA={'home':AUDITORIA_HOME,'baseline_original':BUNDLE['audit']['baseline_original'],
        'reserva_futura_extraida':False,'reserva_futura_inferida':False,
        'test_home_inferido':False,'test_oi20k_inferido':False,'validacao_nunca_vira_treino':True}
    core.save_json(PASTA_RESULTADOS/'auditoria_dados.json',AUDITORIA)
    for p in (PASTA_LOCAL/'home/codigo').rglob('*.py'):
        dest=PASTA_RESULTADOS/'codigo_home'/p.relative_to(PASTA_LOCAL/'home/codigo')
        dest.parent.mkdir(parents=True,exist_ok=True);shutil.copy2(p,dest)
    print('Validações prontas: 456 Home + 1.000 externas; 300 reservadas intactas.')
except Exception as exc:
    registrar_falha(exc,'dados')
    raise


## 5. Congelar a regra antes da inferência

A regra é a mesma do experimento 12: Home deve superar a precisão do baseline,
preservar seu recall e não aumentar omissões altas; a externa deve preservar
as três métricas do candidato 10. Os limites não serão afrouxados após os testes.

Um único valor global deve atender às duas validações. O desempate usa precisão
Home, recall Home, menos omissões altas Home, F1 externa, proximidade de 0,25 e
maior confiança. Qualquer divergência de reprodução em 0,25 impede sugerir limiar.


In [ ]:
PROTOCOLO={'versao':'14-protocolo-global-1.0','grade':list(avaliacao14.THRESHOLDS),
    'parametros_inferencia':avaliacao14.PREDICT_ARGS,'modelo_variavel':False,
    'regra_original_12':BUNDLE['rule'],'regra_12_sha256':EXPECTED['regra_12_sha256'],
    'reproducao_obrigatoria':True,'referencias_reinferidas':False,
    'origem_referencias':'pacote 12 autenticado, baseline e candidato 10 em 0,25',
    'desempate':['maior precisão Home','maior recall Home','menos omissões altas Home',
        'maior F1 externa','mais próximo de 0,25','maior confiança'],
    'limiar_galeria_sem_elegivel':.10,'testes_consultados':False,
    'preenchimento_manual_obrigatorio':False,'substituicao_automatica_de_modelo':False}
PLANO=PASTA_RESULTADOS/'protocolo_congelado_14.json'
core.save_json(PLANO,PROTOCOLO)
FROZEN_FILES.append((PLANO,core.sha256_file(PLANO)))
CONFIG_14={'versao':avaliacao14.VERSION,'experimento':NOME_EXPERIMENTO,
    'expected':EXPECTED,'peso_B_e03_sha256':EXPECTED['B_e03'],'dataset_sha256':EXPECTED['dataset_novo'],
    'politica_sha256':EXPECTED['politica'],'associador_sha256':core.sha256_file(avaliacao_base.__file__),
    'ambiente':AMBIENTE,'ambiente_referencia_12':BUNDLE['config']['ambiente'],
    'protocolo':PROTOCOLO,'semente':SEED,'preenchimento_manual_obrigatorio':False,
    'codigo_embutido_sha256':{n:hashlib.sha256(s.encode()).hexdigest() for n,s in MODULOS_EMBUTIDOS.items()},
    'coortes':{k:len(v) for k,v in COORTES.items()},'novos_treinamentos':0}
CONFIG_14['correcao_cache']={'id': '14-cache-validacao-1.0.1', 'origem': 'Correção da checagem da etapa 6', 'conserva_propostas_numericas': True, 'altera_pesos': False, 'altera_rotulos': False, 'altera_grade': False, 'altera_associador': False, 'reproducao_025_mantida': True}
core.save_json(PASTA_RESULTADOS/'configuracao_diagnostico_14.json',CONFIG_14)
dados12.verify_frozen(FROZEN_FILES)
enviar_resultados('protocolo congelado antes da inferência')
print('Grade e requisitos congelados. Zero formulários para preencher.')


## 6. Inferir o B_e03 uma única vez por imagem

Confiança de captura 0,001, imagem 640, NMS 0,70, máximo de 300 propostas,
FP32 e batch de inferência 1, como no avaliador original. As previsões completas
pós-NMS serão incluídas no pacote final. O código imprime o progresso a cada
100 imagens e salva no Drive ao concluir cada validação.

Se esta célula for repetida na mesma execução, caches completos só serão
reutilizados após conferir pesos, entradas, ambiente e conteúdo.

Correção 1.0.1: propostas numéricas com área zero ou fora dos limites não interrompem o cache; elas são preservadas e registradas em diagnóstico. Estruturas inválidas, valores não finitos, classes ou confiança incompatíveis continuam gerando erro com os valores da proposta.


In [ ]:
CACHE_FILES={};CACHE_METADADOS={};CACHES={}
try:
    MODELO=YOLO(str(PESO_B))
    if {int(k):v for k,v in MODELO.names.items()}!=dict(enumerate(core.NAMES)):
        raise RuntimeError('As classes dos pesos não correspondem às 12 classes do projeto.')
    for coorte,registros in COORTES.items():
        cache,meta=avaliacao14.obtain_cache(MODELO,registros,PASTA_RESULTADOS/'caches'/coorte,
            EXPECTED['B_e03'],AMBIENTE,DEVICE)
        CACHE_FILES[coorte]=cache;CACHE_METADADOS[coorte]=meta
        CACHES[coorte]=avaliacao14.read_cache(cache,registros)
        FROZEN_FILES += [(cache,core.sha256_file(cache)),(cache.parent/'cache_manifesto.json',core.sha256_file(cache.parent/'cache_manifesto.json'))]
        core.save_json(PASTA_RESULTADOS/'caches_concluidos.json',CACHE_METADADOS)
        enviar_resultados('cache concluído: '+coorte)
    del MODELO
    if torch.cuda.is_available():torch.cuda.empty_cache()
    CONFIG_14['caches']=CACHE_METADADOS
    CONFIG_14['imagens_inferidas_nesta_execucao']=sum(
        meta['solicitacao']['imagens'] for meta in CACHE_METADADOS.values() if not meta['reutilizado_nesta_execucao'])
    core.save_json(PASTA_RESULTADOS/'configuracao_diagnostico_14.json',CONFIG_14)
    print('Inferência concluída. A varredura seguinte usa os caches salvos.')
except Exception as exc:
    registrar_falha(exc,'inferência')
    raise


## 7. Reproduzir 0,25 e avaliar a grade automaticamente

Primeiro, verifica os contadores por imagem e os estados dos objetos GT em 0,25.
Em seguida, recalcula associações e alertas para cada confiança da grade.
Isso mede a recuperação real dos objetos, além dos falsos alertas e das omissões;
não pressupõe que toda previsão de confiança baixa se transforme em alerta útil.


In [ ]:
RESULTADOS={};REPRODUCAO=[];REPRO_IMAGENS=[];REPRO_OBJETOS=[]
try:
    for coorte,registros in COORTES.items():
        RESULTADOS[coorte]={}
        base=PASTA_RESULTADOS/'avaliacoes'/coorte
        resultado=avaliacao14.evaluate_cache(registros,CACHES[coorte],.25,politica,
            avaliacao_base.corresponder_objetos,base/avaliacao14.point_name(.25))
        RESULTADOS[coorte][.25]=resultado
        resumo,imagens,objetos=avaliacao14.check_reproduction(
            resultado,BUNDLE['evaluations']['B_e03'][coorte],coorte)
        REPRODUCAO.append(resumo);REPRO_IMAGENS+=imagens;REPRO_OBJETOS+=objetos
        print(coorte,'| reprodução em 0,25:',resumo['reproduziu'],flush=True)
        for conf in avaliacao14.THRESHOLDS:
            if conf==.25:continue
            RESULTADOS[coorte][conf]=avaliacao14.evaluate_cache(registros,CACHES[coorte],conf,
                politica,avaliacao_base.corresponder_objetos,base/avaliacao14.point_name(conf))
            a=RESULTADOS[coorte][conf]['summary']['alertas_por_objeto']
            print(coorte,'| confiança',conf,'| TP',a['tp'],'FN',a['fn'],'FP',a['fp'],flush=True)
        core.save_json(PASTA_RESULTADOS/'reproducao_025_resumo.json',REPRODUCAO)
        core.save_csv(PASTA_RESULTADOS/'reproducao_025_por_imagem.csv',REPRO_IMAGENS)
        core.save_csv(PASTA_RESULTADOS/'reproducao_025_por_objeto.csv',REPRO_OBJETOS)
        enviar_resultados('grade avaliada: '+coorte)
    REPRODUZIU=all(r['reproduziu'] for r in REPRODUCAO)
    SELECAO=avaliacao14.select_threshold(RESULTADOS,BUNDLE['rule'],REPRODUZIU)
    print('Resultado da seleção:',SELECAO['status'])
    print('Limiar candidato nas validações:',SELECAO['limiar_sugerido'])
except Exception as exc:
    registrar_falha(exc,'avaliação da grade')
    raise


## 8. Gerar o relatório e o pacote final

A comparação inclui os valores absolutos e as diferenças de FN/FP em relação
ao B em 0,25. Os CSVs identificam quais perdas em relação ao candidato 10 foram
recuperadas em cada limiar e os casos ainda detectados cujo alerta mudou por
prioridade/geometria. A galeria é montada automaticamente, sem fichas.

Se nenhum ponto atender a todos os requisitos, o relatório registra o resultado
negativo. Uma configuração elegível é apenas candidata nas validações; nenhum
modelo em uso é substituído e nenhum teste ou imagem reservada é aberto.


In [ ]:
try:
    dados12.verify_frozen(FROZEN_FILES)
    RESUMO=relatorios14.build_report(PASTA_RESULTADOS,RESULTADOS,BUNDLE,SELECAO,
        REPRODUCAO,COORTES,CACHES,politica,CONFIG_14)
    core.save_json(PASTA_RESULTADOS/'integridade_final.json',{
        'conferida':True,'arquivos_conferidos':len(FROZEN_FILES),
        'rotulos_alterados':False,'pesos_originais_preservados':True,
        'reserva_futura_extraida':False,'reserva_futura_inferida':False,
        'novos_treinamentos':0,'preenchimento_manual_obrigatorio':False})
    # Uma falha anterior resolvida fica no histórico, fora do estado atual.
    erro_anterior=PASTA_RESULTADOS/'execucao_interrompida.json'
    if erro_anterior.is_file():
        historico=PASTA_RESULTADOS/'historico_falhas';historico.mkdir(exist_ok=True)
        erro_anterior.replace(historico/('execucao_interrompida_'+datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S_%f')+'.json'))
    PACOTE_FINAL=relatorios14.package_reports(PASTA_RESULTADOS)
    enviar_resultados('pacote final do diagnóstico 14')
    print(RESUMO['mensagem'])
    print('Diagnóstico finalizado. Arquivo de resultados:',PACOTE_FINAL)
    display(FileLink(str(PACOTE_FINAL)))
    display(FileLink(str(PASTA_RESULTADOS/'relatorio_diagnostico_14.html')))
    display(FileLink(str(PASTA_RESULTADOS/'comparacao_confianca.csv')))
except Exception as exc:
    registrar_falha(exc,'relatório e salvamento')
    raise
